# Setup and function definition

In [ ]:
import os
import pandas as pd
import numpy as np
from src.utils import load_data, compute_fourier, geom_rank_cell_nodes, get_firingrate_hist,extract_main_cell_types, get_estim_data, get_plv, get_ppc
from src.plotting import plot_fourier_analysis, psth_plot
from src.plotting_old import psth_plot_old

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
%matplotlib tk

In [ ]:
import matplotlib
matplotlib.rcParams.update({'font.size': 12})

Look at the data.csv as a dataframe

In [ ]:
simulations = pd.read_csv("data_updated.csv")
pd.options.display.max_columns = None

In [ ]:
#network : "../../../../Downloads/O1_data/circuit_config.json",
# import json
# from pathlib import Path

# data_dir = Path("data")
# #input the position of O1_data in your directory
# new_network = "../../../../Downloads/O1_data/circuit_config.json"

# for folder in data_dir.iterdir():
#     if folder.is_dir():
#         json_file = folder / "sim_with_stim.json"

#         if json_file.exists():
#             with open(json_file, "r") as f:
#                 config = json.load(f)

#             config["network"] = new_network

#             with open(json_file, "w") as f:
#                 json.dump(config, f, indent=4)

#             print(f"Updated: {json_file}")

In [ ]:
# simulations["source_file"] = simulations["source_file"].str.replace(
#     r"^(?:\.\./Documents/eStim/|(?:\.\./){4}e/tACS/)",
#     "../../../e/tACS/",
#     regex=True,
# )

# simulations.loc[simulations["sim_id"] == "baseline", "source_file"] = (
#     "../../../e/tACS/weji/sim0_CoreNeuron/sim_with_stim.json"
# )

# simulations["sim_id"] = simulations["sim_id"].str.replace(
#     r"^sim",
#     "",
#     regex=True,
# )

#I am working in wsl so I need to adjust all the config paths

# simulations["source_file"] = simulations["source_file"].str.replace(
#     r"^../../../",
#     "/mnt/",
#     regex=True,
# )

In [ ]:
simulations

In [ ]:
simulations.to_csv('data_updated.csv')

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12})

def getSpiking(data,cellType=None):

    config_path = data['source_file'].values[0]

    simulationData, allData, cellData, spikeData = load_data(config_path)

    if cellType is None:
        return spikeData
    else:
        cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

        return spikeData, cellData, cell_to_rank

def goertzel_component(hist: np.ndarray, sample_period_s: float, freq: float) -> tuple[complex, float]:
    """
    Exact-frequency complex Fourier coefficient, evaluated directly at `freq`
    rather than snapping to the nearest FFT bin from compute_fourier.

    Eliminates spectral leakage whenever hist.shape[0] * freq * sample_period_s
    isn't an integer -- i.e. whenever `freq` doesn't land exactly on an FFT bin
    for this record length. Leakage distorts both amplitude and (more severely)
    phase, so relying on the nearest bin silently produces wrong phase values
    whenever bin_width/time_stop don't happen to make `freq` an exact bin.

    Returns (coeff, amp) on the same raw/unnormalized scale as compute_fourier's
    yf and amplitude, so this is a drop-in replacement for yf[idx_pos] /
    amplitude[idx_pos].
    """
    n = np.arange(hist.shape[0])
    t = n * sample_period_s
    w = 2 * np.pi * freq
    coeff = np.sum(hist * np.exp(-1j * w * t))
    amp = np.abs(coeff) / hist.shape[0]
    return coeff, amp

def getBaselineData(data, targetFreq, cellType=None, delay=1500):

    config_path = data['source_file'].values[0]

    simulationData, allData, cellData, spikeData = load_data(config_path)
    cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

    if cellType is not None:
        cell_types = extract_main_cell_types(spikeData, cellData, cell_to_rank)
        spikeData, cell_to_rank = cell_types[cellType]

    hist, _ = get_firingrate_hist(
                simulationData, spikeData, bin_width=5, n_neurons=cell_to_rank.shape[0], delay=delay , time_stop=2500, smoothed=True)

    if delay==1500:
    
        xf_pos, amplitude, yf = compute_fourier(hist, 5 * 1e-3)
    
        return xf_pos, amplitude, yf

    else:

        coeff, amp_at_freq = goertzel_component(hist, 5 * 1e-3, targetFreq)

        coeff = coeff * np.exp(-1j * (np.pi/2 + np.pi))

        return coeff, amp_at_freq

def getBaselineComplex(data, targetFreq,cellType=None,delay=1500):

    if delay == 1500:

        xf_pos, amplitude, yf = getBaselineData(data, targetFreq,cellType, delay)

        return yf[np.argmin(np.abs(xf_pos-targetFreq))]

    else: 

        coeff, amp_at_freq = getBaselineData(data, targetFreq, cellType, delay)

        return coeff

def getBaselinePower(data, targetFreq,cellType=None):

    xf_pos, amplitude, yf = getBaselineData(data, targetFreq,cellType)

    normalized_amp = amplitude #/ amplitude[0]

    power = normalized_amp ** 2
    power_norm = power / np.sum(power)

    return power_norm[np.argmin(np.abs(xf_pos-targetFreq))]

def _windowed_spike_phases(
    spike_data,
    t_stop,
    freq,
    phase0=0.0,
    phase_orientation=0.0,
    stim_start=0.0,
    delay=0.0,
) -> np.ndarray:
    spike_times_s = np.array(spike_data.index.tolist()) * 1e-3
    
    spike_times_s = spike_times_s[spike_times_s > delay]
    
    spike_times_s = spike_times_s[spike_times_s < t_stop]
    
    return 2 * np.pi * freq * (spike_times_s - stim_start) + phase0 + phase_orientation

def getBaselinePLV(data, targetFreq,cellType=None):

    if cellType is not None:
        spikeData, cellData, cell_to_rank = getSpiking(data,cellType)
    else:
        spikeData = getSpiking(data)

    if cellType is not None:
        cell_types = extract_main_cell_types(spikeData, cellData, cell_to_rank)
        spikeData, cell_to_rank = cell_types[cellType]

    theta = _windowed_spike_phases(
        spikeData,
        2500*1e-3,
        targetFreq,
        delay=1500*1e-3,
        stim_start=1500*1e-3
    )
    if len(theta) == 0:
        return np.nan
    return np.abs(np.mean(np.exp(1j * theta)))

In [ ]:
def getPhases_from_h5(simName):

    simFile = simulations[simulations['sim_id']==simName]['source_file'].values[0]
    
    simulationData, allData, cellData, spikeData = load_data(simFile)
    Efields, delay, duration, rampup, rampdown = get_estim_data(
        simulationData
    )

    if len(Efields) == 0:
        raise ValueError(f"No Efields found in {sc_file!r}")

    # Your original loop effectively used the last Efield.
    # Keep that behavior, but allow selecting a different field.
    Efield = Efields[-1]

    freq = Efield["frequency"]
    phase0 = Efield["phase"]
    phases = get_phase(spikeData, freq, delay, phase0)

    return phases

In [ ]:
def plot_condition_means(df,baseline, ax, color, id_col='stim_amp', z_col='complex',max_amp=None):
    z = df[z_col].astype(complex)

    if max_amp is not None:
        z = z[df['stim_amp']<=max_amp]
    
    means = z.groupby(df[id_col]).mean().sort_index()   # sorted by condition id
    zs = np.insert(means.values, 0, baseline)
    ax.plot(np.angle(zs), np.abs(zs), '-', c=color, lw=2, ms=5)
    return means

In [ ]:
def plot_linear_fit(x,y,ax,color='k',label=None):

    from scipy import stats
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(x,y)
    r_squared = r_value**2
    
    ax.plot(np.array([x[0],x[-1]]),np.array([slope*x[0]+intercept,x[-1]*slope+intercept]),c=color,label=label)
    
    print(f"R^2 = {r_squared:.4f}")

def plot_condition_means_linear(df, baseline, ax, color, id_col='stim_amp', z_col='plv', label=None):
    means = df.groupby(id_col)[z_col].mean().sort_index()
    xs = np.insert(means.index.values.astype(float), 0, 0.0)
    ys = np.insert(means.values, 0, baseline)
    ax.plot(xs, ys, '-', c=color, lw=2, ms=5, label=label)
    return means

# --- marker size mapping -------------------------------------------------
BASE_SIZE = 0      # size of the baseline (amp = 0) point
SCALE     = 20      # points^2 per unit of stim_amp

def marker_sizes(amps):
    """Map stimulation amplitudes to marker areas, keeping amp=0 visible."""
    return BASE_SIZE + np.asarray(amps, dtype=float) * SCALE

def polar_scatter(ax, df, baseline_complex=None, color='k', label=None,z_col='complex',max_amp=None,marker='o'):
    z = df[z_col].astype(complex)
    amps = df['stim_amp']

    if max_amp is not None:
        z = z[amps<=max_amp]
        amps = amps[amps<=max_amp]

    

    if baseline_complex is not None:
        theta = np.insert(np.angle(z), 0, np.angle(baseline_complex))
        r     = np.insert(np.abs(z).values, 0, np.abs(baseline_complex))
        amps = np.insert(amps,0,0)
    else:
        theta = np.angle(z)
        r = np.abs(z)
        
    ax.scatter(theta, r,
               marker=marker,
               c=color,
               label=label,
               s=marker_sizes(amps),
               edgecolors='none',
               alpha=0.85,
               zorder=3)

In [ ]:
def rayleigh_test(alpha):
    """Rayleigh's z test for circular uniformity. alpha in radians."""
    alpha = np.asarray(alpha, dtype=float)
    n = alpha.size
    R = np.abs(np.exp(1j * alpha).sum())      # resultant length (unnormalised)
    Rbar = R / n
    z = n * Rbar ** 2
    # Zar's approximation to the exact p-value
    p = np.exp(np.sqrt(1 + 4 * n + 4 * (n ** 2 - R ** 2)) - (1 + 2 * n))
    return {"z": z, "p": min(p, 1.0), "Rbar": Rbar, "n": n}


import numpy as np

def _kuiper_pvalue(lam, n_terms=100):
    """Asymptotic Q_K(lambda) for the Kuiper statistic."""
    j = np.arange(1, n_terms + 1)
    p = 2 * np.sum((4 * j ** 2 * lam ** 2 - 1) * np.exp(-2 * j ** 2 * lam ** 2))
    return float(np.clip(p, 0.0, 1.0))


import numpy as np
import pandas as pd


def preferred_phases(df, phase_col="phase_rad", cell_col="cell_id",
                     min_spikes=1, min_plv=None, plv_col="plv"):
    """Collapse a spike-level frame to one preferred phase per cell."""
    g = df.groupby(cell_col)[phase_col]
    z = g.apply(lambda p: np.exp(1j * np.asarray(p, dtype=float)).mean())
    out = pd.DataFrame({
        "mean_phase": np.mod(np.angle(z), 2 * np.pi),
        "plv": np.abs(z),
        "n_spikes": g.size(),
    })
    out = out[out["n_spikes"] >= min_spikes]
    if min_plv is not None:
        col = plv_col if plv_col in df.columns else "plv"
        source = df.groupby(cell_col)[col].first() if col in df.columns else out["plv"]
        out = out[source.reindex(out.index) >= min_plv]
    return out


def kuiper_test(alpha, beta=None, by_cell=False, n_terms=100,
                permutations=None, seed=None, **cell_kwargs):
    """Kuiper's V test on circular data (radians).

    alpha, beta : array of phases, or — when by_cell=True — a DataFrame with
                  a phase column and a cell-id column.
    by_cell     : collapse each input to one preferred phase per cell before
                  testing, so cells rather than spikes are the sampling unit.
    cell_kwargs : forwarded to preferred_phases (phase_col, cell_col,
                  min_spikes, min_plv, plv_col).
    """
    def _prep(x):
        if by_cell:
            if not isinstance(x, pd.DataFrame):
                raise TypeError("by_cell=True requires a DataFrame input")
            pc = preferred_phases(x, **cell_kwargs)
            return np.mod(pc["mean_phase"].to_numpy(), 2 * np.pi), len(pc)
        return np.mod(np.asarray(x, dtype=float).ravel(), 2 * np.pi), None

    alpha, n_cells_a = _prep(alpha)
    beta, n_cells_b = (_prep(beta) if beta is not None else (None, None))

    if beta is None:
        n = alpha.size
        u = np.sort(alpha) / (2 * np.pi)
        i = np.arange(1, n + 1)
        V = np.max(i / n - u) + np.max(u - (i - 1) / n)
        n_eff = n
        out = {"V": V, "n": n, "test": "one-sample"}
    else:
        n1, n2 = alpha.size, beta.size
        V = _two_sample_V(alpha, beta)
        n_eff = n1 * n2 / (n1 + n2)
        out = {"V": V, "n1": n1, "n2": n2, "test": "two-sample"}

    out["unit"] = "cell" if by_cell else "spike"

    if permutations:
        rng = np.random.default_rng(seed)
        null = np.empty(permutations)
        if beta is None:
            for k in range(permutations):
                s = np.sort(rng.uniform(0, 2 * np.pi, alpha.size)) / (2 * np.pi)
                i = np.arange(1, s.size + 1)
                null[k] = np.max(i / s.size - s) + np.max(s - (i - 1) / s.size)
        else:
            pooled = np.concatenate([alpha, beta])
            for k in range(permutations):
                p = rng.permutation(pooled)
                null[k] = _two_sample_V(p[:alpha.size], p[alpha.size:])
        out["p"] = (np.sum(null >= V) + 1) / (permutations + 1)
        out["method"] = f"permutation ({permutations})"
    else:
        lam = V * (np.sqrt(n_eff) + 0.155 + 0.24 / np.sqrt(n_eff))
        out["p"] = _kuiper_pvalue(lam, n_terms)
        out["method"] = "asymptotic"

    return out


def _two_sample_V(x, y):
    """Kuiper statistic between two empirical CDFs."""
    xs, ys = np.sort(x), np.sort(y)
    grid = np.concatenate([xs, ys])
    cdf1 = np.searchsorted(xs, grid, side="right") / xs.size
    cdf2 = np.searchsorted(ys, grid, side="right") / ys.size
    return np.max(cdf1 - cdf2) + np.max(cdf2 - cdf1)

def get_phase(spikes, FREQ_HZ = 10.0, T0_MS = 1500.0, phase0 = 0.0):
    
    PERIOD_MS = 1000.0 / FREQ_HZ  # 100 ms

    spikes = spikes[spikes.index >= T0_MS]
    
    t = spikes.index.to_numpy(dtype=float)
    
    # Wrapped to [0, 2π). Negative values (spikes before t0) wrap correctly.
    phase_rad = ((2 * np.pi * (t - T0_MS) / PERIOD_MS)+phase0) % (2 * np.pi)
    
    result = pd.DataFrame(
        {
            "cell_id": spikes.to_numpy(),
            "phase_rad": phase_rad,
            "phase_deg": np.degrees(phase_rad),
        },
        index=spikes.index,
    )
    

    return result

def circ_stats(p):
    z = np.exp(1j * p).mean()
    return pd.Series({"mean_phase": np.angle(z) % (2 * np.pi), "plv": np.abs(z)})

from matplotlib.colors import is_color_like

In [ ]:
def _darken(color, amount=0.6):
    """Return `color` scaled toward black. amount<1 darkens, amount=1 is unchanged."""
    import colorsys
    from matplotlib.colors import to_rgb
    r, g, b = to_rgb(color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0.0, min(1.0, l * amount))
    return colorsys.hls_to_rgb(h, l, s)

In [ ]:
def plot_rose(results, N_BINS=36, labels=None, normalize=True, style="bar",
              color=None, plv_scale=0.5, hist_fill=0.9,
              title=None,
              ax=None, legend=True, figsize=(2.75, 2.75)):
    """
    plv_scale : PLV value that reaches the outer edge of the polar axis.
                Fixed at 0.5 -- the *line* radius is a literal, comparable
                PLV scale, so a PLV of 0.5 always reaches the rim.
    hist_fill : fraction of the outer edge that the tallest histogram bar
                (across every dataset in this call) should reach. Purely a
                display choice -- bar height encodes relative shape only,
                not a literal fraction value, and is independent of plv_scale.
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import is_color_like

    # --- accept one DataFrame, a list/tuple, or a dict of {label: DataFrame} ---
    if isinstance(results, dict):
        items = list(results.items())
    elif isinstance(results, (list, tuple)):
        items = [(f"set {i}", r) for i, r in enumerate(results, 1)]
    else:
        items = [("", results)]
    if labels is not None:
        items = [(lab, r) for lab, (_, r) in zip(labels, items)]

    edges = np.linspace(0, 2 * np.pi, N_BINS + 1)
    width = np.diff(edges)
    centers = edges[:-1] + width / 2

    if ax is None:
        _, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=figsize)

    # --- resolve colors: None -> tab10, single color, sequence, or {label: color} ---
    default = plt.get_cmap("tab10")(np.arange(len(items)) % 10)
    if color is None:
        colors = default
    elif is_color_like(color):
        colors = [color] * len(items)
    elif isinstance(color, dict):
        colors = [color.get(lab, d) for (lab, _), d in zip(items, default)]
    else:
        colors = list(color)
        if len(colors) < len(items):
            colors = [colors[i % len(colors)] for i in range(len(items))]

    # --- pass 1: histograms and mean vectors ---
    hists, zs, ns = [], [], []
    for _, res in items:
        phase = np.radians(np.asarray(res["phase_deg"], dtype=float))
        counts, _ = np.histogram(phase, bins=edges)
        h = counts / counts.sum() if normalize and counts.sum() else counts.astype(float)
        hists.append(h)
        zs.append(np.exp(1j * phase).mean() if phase.size else 0j)
        ns.append(phase.size)

    # bars: rescaled for DISPLAY ONLY -- tallest bar (across all datasets)
    # reaches hist_fill * plv_scale, regardless of its literal fraction value
    rmax = max((h.max() for h in hists if h.size), default=0.0)
    if rmax > 0:
        hists_plot = [h / rmax * hist_fill * plv_scale for h in hists]
    else:
        hists_plot = hists

    # --- pass 2: draw ---
    for (label, _), color_i, h_plot, z, n in zip(items, colors, hists_plot, zs, ns):
        legend_label = f"{label} (n={n})" if label else f"n={n}"
        if style == "bar":
            ax.bar(centers, h_plot, width=width, bottom=0, color=color_i,
                   edgecolor=color_i, linewidth=0.5, alpha=0.45, label=legend_label)
        else:
            ax.plot(np.append(centers, centers[0]), np.append(h_plot, h_plot[0]),
                    color=color_i, lw=1.8, alpha=0.9, label=legend_label)

        angle = float(np.angle(z))
        plv = float(np.abs(z))
        if plv > plv_scale:
            import warnings
            warnings.warn(f"PLV={plv:.3f} exceeds plv_scale={plv_scale}; line will be clipped")

        line_color = _darken(color_i, 0.6)
        ax.plot([angle, angle], [0, plv], color=line_color, lw=2,
                solid_capstyle="round", zorder=3)  # raw plv -- axis top IS plv_scale

    ax.set_ylim(0, plv_scale)
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    if title:
        ax.set_title(title, pad=12)
    if len(items) > 1 and legend:
        ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.05), frameon=False)
    return ax

# Intro figure

In [ ]:
simulationData, allData, cellData, spikeData = load_data('data/sim7Hz1Vm_seed2/sim_with_stim.json')

In [ ]:
cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

In [ ]:
psth_plot_old(spikeData, title, cell_to_rank, simulationData, cellData.shape[0], bin_width=5, smoothed=True, save_histogram='fig1.png')

# Results

In [ ]:
# color guide 
# plot each color with the label underneath to get an idea of the colors

In [ ]:
peru = '#CD853F'
pink = '#CC79A7'
darkgrey = '#333333'
lighterpink = '#FFA3D7'
lighterorange = '#FFB347'
midgrey = '#7F7F7F'

darkbrown = '#994F00'
grey = 'slategrey'

blue = '#0072B2'
red = '#d62728'
purple = '#9467bd'
magenta = '#FF00FF'
lightblue = '#56B4E9'

In [ ]:
baseline = simulations[simulations['sim_id']=='baseline']

## Single column reproduces behavior of larger network

In [ ]:
#Choose a simulation you want to look at
sim_id = 'hex0_10Hz_4Vm_OBI'
#choose bin size for firing rate calculation
bin = 5
#get the location of the config file for that simulation
config_path = simulations.loc[simulations['sim_id'] == sim_id, 'source_file'].values[0]
freq = simulations.loc[simulations['sim_id'] == sim_id, 'stim_freq'].values[0]
delay = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_start'].values[0]
time_stop = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_stop'].values[0]
#load the data for that simulation
print(f"Loading data for simulation {sim_id} from {config_path}")
output_folder = os.path.join("output", sim_id)
os.makedirs(output_folder, exist_ok=True)

In [ ]:
simulationData, allData, cellData, spikeData = load_data('/mnt/e/tACS/fromOBI/hex0_10Hz_4Vm/sim_with_stim.json')
cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

In [ ]:
#Choose a simulation you want to look at
sim_id = 'O1_10Hz_4Vm_obi'
#choose bin size for firing rate calculation
bin = 5
#get the location of the config file for that simulation
config_path = simulations.loc[simulations['sim_id'] == sim_id, 'source_file'].values[0]
freq = simulations.loc[simulations['sim_id'] == sim_id
    , 'stim_freq'].values[0]
delay = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_start'].values[0]
time_stop = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_stop'].values[0]

simulationData_lc, allData_lc, cellData_lc, spikeData_lc = load_data(config_path)
cell_to_rank_lc = geom_rank_cell_nodes(cellData_lc, simulationData_lc)

In [ ]:
multiple_psth_plot([spikeData_lc,spikeData], "", [cell_to_rank_lc,cell_to_rank], [simulationData_lc,simulationData],['O1','hex0'], bin_width=bin, save_histogram=os.path.join(output_folder, f"psth_{sim_id}_{bin}bin_smoothed10.png"), figsize=(6,3),colorList = [darkbrown, grey], normalized=True)

In [ ]:
#statistical analysis of the
# two firing rate -> is it significantly different accros time?

In [ ]:
#I need to obtain hex0 and O1 which contain the PLV values for 10Hz
simulations[(simulations['deployment'] == 'OBI') | (simulations['deployment'] == 'obi')]
O1_source_files = simulations.loc[simulations["sim_id"].str.startswith("O1", na=False), "source_file"]
hex0_source_files = simulations.loc[simulations["sim_id"].str.startswith("hex0", na=False), "source_file"]

In [ ]:
def average_metrics_by_amp(
    source_files,
    stim_amps=(0.25, 1, 4),
    efield_index=-1,
):
    stim_amps = list(stim_amps)

    # Store individual seed results separately for each amplitude.
    grouped = {
        amp: {"plv": [], "ppc": []}
        for amp in stim_amps
    }

    for sc_file in source_files:
        simulationData, allData, cellData, spikeData = load_data(sc_file)
        Efields, delay, duration, rampup, rampdown = get_estim_data(
            simulationData
        )

        if len(Efields) == 0:
            raise ValueError(f"No Efields found in {sc_file!r}")

        # Your original loop effectively used the last Efield.
        # Keep that behavior, but allow selecting a different field.
        Efield = Efields[efield_index]

        amp = np.round(
            np.sqrt(
                Efield["Ex"] ** 2
                + Efield["Ey"] ** 2
                + Efield["Ez"] ** 2
            ),
            2,
        )

        # Skip amplitudes outside the requested list.
        if amp not in grouped:
            continue

        freq = Efield["frequency"]
        phase0 = Efield["phase"]
       
        # Calculate the metrics once per simulation/seed.
        plv = get_plv(spikeData, simulationData.time_stop, freq, phase0, delay)
        ppc = get_ppc(spikeData, simulationData.time_stop, freq, phase0, delay)

        grouped[amp]["plv"].append(plv)
        grouped[amp]["ppc"].append(ppc)

    result = {
        "stim_amp": stim_amps,
        "plv": [],
        "ppc": [],
        "n_seeds": [],
    }

    # Build the output in amplitude order, not source-file order.
    for amp in stim_amps:
        for metric in ("plv", "ppc"):
            values = grouped[amp][metric]

            result[metric].append(
                np.mean(values, axis=0) if values else np.nan
            )

        result["n_seeds"].append(len(grouped[amp]["plv"]))

    return result


O1 = average_metrics_by_amp(O1_source_files)
hex0 = average_metrics_by_amp(hex0_source_files)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,4))
ax.plot(hex0['stim_amp'], hex0['ppc'], 'o',c=grey)
ax.plot(O1['stim_amp'], O1['ppc'], 'o',c=darkbrown)
# ax.set_xscale('log')
ax.set_xlabel('Stimulus\namplitude (V/m)')
ax.set_ylabel('PPC')
ax.legend(['hex0','O1'], loc='lower right')
plt.tight_layout()
plt.savefig('O1_vs_hex0_ppc.png',dpi=600)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,4))
ax.plot(hex0['stim_amp'], hex0['plv'], 'o',c=grey)
ax.plot(O1['stim_amp'], O1['plv'], 'o',c=darkbrown)
# ax.set_xscale('log')
ax.set_xlabel('Stimulus\namplitude (V/m)')
ax.set_ylabel('PLV')
ax.legend(['hex0','O1'], loc='lower right')
plt.tight_layout()
plt.savefig('O1_vs_hex0_plv.png',dpi=600)

#make the image prettier -> think about the colors

### Plots figure 2c-2e, and runs statistical tests for differences between phase preferences of hex0 vs O1

In [ ]:
spikePhasesO1_025 = getPhases_from_h5('O1_10Hz0.25Vm_obi')
spikePhasesO1_1a = getPhases_from_h5('O1_10Hz_1Vm_OBI')
spikePhasesO1_1b = getPhases_from_h5('O1_10Hz1Vm_OBI_2')
spikePhasesO1_1c = getPhases_from_h5('O1_10Hz_1Vm_OBI_3')
spikePhasesO1_4 = getPhases_from_h5('O1_10Hz_4Vm_obi')

spikePhasesO1_1 = pd.concat([spikePhasesO1_1a,spikePhasesO1_1b,spikePhasesO1_1b])

spikePhasesHex0_025a = getPhases_from_h5('hex0_10Hz0.25Vm_obi')
spikePhasesHex0_025b = getPhases_from_h5('hex0_10Hz0.25Vm_seed2')
spikePhasesHex0_1 = getPhases_from_h5('hex0_10Hz1Vm_OBI')
spikePhasesHex0_4 = getPhases_from_h5('hex0_10Hz_4Vm_OBI')

spikePhasesHex0_025 = pd.concat([spikePhasesHex0_025a,spikePhasesHex0_025b])


In [ ]:
plot_rose([spikePhasesHex0_025,spikePhasesO1_025],legend=False,color=[grey,darkbrown]).get_figure().savefig("O1vHex0_025.png", dpi=300, bbox_inches="tight")


In [ ]:
kuiper_test(spikePhasesHex0_025['phase_rad'],spikePhasesO1_025['phase_rad'])

In [ ]:
plot_rose([spikePhasesHex0_1,spikePhasesO1_1],legend=False,color=[grey,darkbrown]).get_figure().savefig("O1vHex0_1.png", dpi=300, bbox_inches="tight")

In [ ]:
kuiper_test(spikePhasesHex0_1['phase_rad'],spikePhasesO1_1['phase_rad'])

In [ ]:
plot_rose([spikePhasesHex0_4,spikePhasesO1_4],legend=False,color=[grey,darkbrown]).get_figure().savefig("O1vHex0_4.png", dpi=300, bbox_inches="tight")

In [ ]:
kuiper_test(spikePhasesHex0_4['phase_rad'],spikePhasesO1_4['phase_rad'])

# tACS entrains network activity

In [ ]:
#S.3 avg FR for all frequencies and amplitudes


#statistical testing that there is no significant differnece between avg FR baseline and avg FR stimulated -> t-tests?


# And now we plot figure S1

In [ ]:
#Choose a simulation you want to look at
sim_id = 'baseline'
#choose bin size for firing rate calculation
bin = 5
#get the location of the config file for that simulation
config_path = simulations.loc[simulations['sim_id'] == sim_id, 'source_file'].values[0]
freq = simulations.loc[simulations['sim_id'] == sim_id, 'stim_freq'].values[0]
delay = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_start'].values[0]
time_stop = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_stop'].values[0]
#load the data for that simulation
print(f"Loading data for simulation {sim_id} from {config_path}")
simulationData, allData, cellData, spikeData = load_data(config_path)
cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)
output_folder = os.path.join("output", sim_id)
os.makedirs(output_folder, exist_ok=True)

In [ ]:
from scipy.fft import fft, fftfreq

def compute_fourier(hist, T):
    """Return one-sided FFT quantities: frequencies, amplitude, phase, raw coefficients."""
    yf = fft(hist)
    N = hist.shape[0]
    xf = fftfreq(N, T)

    pos = slice(0, N // 2)
    xf_pos = xf[pos]

    amplitude = 2.0 / N * np.abs(yf[pos])
    phase = np.angle(yf[pos])

    return xf_pos, amplitude, phase, yf

def plot_power_spectrum(xf_pos, amplitude, freq, idx_base=None):
    """Single-panel plot: total-power-normalised power spectrum."""
    amplitude = amplitude / amplitude[0]
    power = amplitude ** 2
    power_norm = power / np.sum(power)

    # if idx_base is None:
    #     idx_base = np.argmax(power_norm)

    fig, ax = plt.subplots(figsize=(6, 4))

    ax.stem(xf_pos[1:], power_norm[1:], basefmt=" ")
    ax.set(title='Normalized Power Spectrum', xlabel='Frequency (Hz)', ylabel='Power / total')
    ax.grid(True)

    if freq > 0:
        idx_pos = np.argmin(np.abs(xf_pos - freq))
        ax.stem(xf_pos[idx_pos], power_norm[idx_pos], markerfmt='ro', basefmt=" ")

    if idx_base is not None:
        ax.stem(xf_pos[idx_base], power_norm[idx_base], markerfmt='go', basefmt=" ")
    ax.set_xlim([0,20])
    _add_legend(fig, freq, xf_pos, idx_base)
    return fig


def _add_legend(fig, freq, xf_pos, idx_base):
    """Shared legend helper for stimulation and baseline frequency markers."""
    from matplotlib.lines import Line2D
    legend_handles = []
    if freq > 0:
        legend_handles.append(Line2D([], [], color='red', marker='o', linestyle='None',
                                     label=f'Stimulation freq ({freq} Hz)'))
    if idx_base is not None:
        legend_handles.append(Line2D([], [], color='green', marker='o', linestyle='None',
                                    label=f'Max baseline power ({xf_pos[idx_base]:.1f} Hz)'))
    fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=True, fontsize=9)
    fig.tight_layout(rect=[0, 0.04, 1, 1])


In [ ]:
hist, _ = get_firingrate_hist(
        simulationData, spikeData, bin_width=bin,
        n_neurons=cell_to_rank_lc.shape[0], delay=200 , time_stop=2500, smoothed=True
    )

In [ ]:
xf_pos, amplitude, phase, yf = compute_fourier(hist, bin * 1e-3)

In [ ]:
#Choose a simulation you want to look at
sim_id = 'baseline_highCorrelation_stopStart'
#choose bin size for firing rate calculation
bin = 5
#get the location of the config file for that simulation
config_path = simulations.loc[simulations['sim_id'] == sim_id, 'source_file'].values[0]
freq = simulations.loc[simulations['sim_id'] == sim_id
    , 'stim_freq'].values[0]
delay = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_start'].values[0]
time_stop = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_stop'].values[0]
simulationData_lc, allData_lc, cellData_lc, spikeData_lc = load_data(config_path)
print(simulationData_lc.time_stop)
cell_to_rank_lc = geom_rank_cell_nodes(cellData_lc, simulationData_lc)

In [ ]:
hist_baseline, _ = get_firingrate_hist(
        simulationData_lc, spikeData_lc, bin_width=bin,
        n_neurons=cell_to_rank_lc.shape[0], delay=200 , time_stop=500, smoothed=True
    )
xfb, amplitude_baseline, phase_baseline, yfb = compute_fourier(hist_baseline, bin * 1e-3)

In [ ]:
#plot normalized power -> DC normalized?
plot_power_spectrum(xf_pos, amplitude, 0)

In [ ]:
#plot normalized power -> DC normalized?
plot_power_spectrum(xfb, amplitude_baseline, 0)

# Here, we create figure 3

In [ ]:
# create a dictionary where I have per amplitude key (0.25, 0.5, 1,2,4) a list of 3 elements which is the source_file name, which can be another list if there multiple seed for the same stimulation type
#these are all the simulations which don't have highCorrelation, replay, hex0, O1 and delay in the sim_id 

In [ ]:
#S.3 avg FR for all frequencies and amplitudes
#load the simulations make firing rates of the stimulate part and calculate average and then plot the avg FR as a histogram agains the baseline FR
#statistical testing that there is no significant differnece between avg FR baseline and avg FR stimulated -> t-tests?

In [ ]:
simulations

In [ ]:
data = simulations[~simulations['sim_id'].str.contains('delay')]
data = data[~data['sim_id'].str.contains('highCorrelation')]
data = data[~data['sim_id'].str.contains('obi', case=False, na=False)]
data = data[~data['sim_id'].str.contains('hex0', case=False, na=False)]
data = data[~data['sim_id'].str.contains('baseline', case=False, na=False)]
data = data[~data['sim_id'].str.contains('replay', case=False, na=False)]

data.sort_values('stim_amp',inplace=True)

In [ ]:
baseline = simulations[simulations['sim_id']=='baseline']
baselineComplex7 = getBaselineComplex(baseline,7)
baselineComplex10 = getBaselineComplex(baseline,10)
baselineComplex15 = getBaselineComplex(baseline,15)
bplv10 = getBaselinePLV(baseline,10)
bplv7 = getBaselinePLV(baseline,7)
bplv15 = getBaselinePLV(baseline,15)


In [ ]:
data7 = data[data['stim_freq']==7]
data10 = data[data['stim_freq']==10]
data15 = data[data['stim_freq']==15]

### Figure 3a

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,4), subplot_kw={'projection': 'polar'},layout='constrained')

angle7 = np.insert(np.angle(data7['complex'].astype(complex)), 0, np.angle(baselineComplex7))
mag7 = np.insert(np.abs(data7['complex'].astype(complex)).values,0,np.abs(baselineComplex7))
amplitudes7 = np.insert(data7['stim_amp'],0,0)

angle10 = np.insert(np.angle(data10['complex'].astype(complex)), 0, np.angle(baselineComplex10))
mag10 = np.insert(np.abs(data10['complex'].astype(complex)).values,0,np.abs(baselineComplex10))
amplitudes10 = np.insert(data10['stim_amp'],0,0)

angle15 = np.insert(np.angle(data15['complex'].astype(complex)), 0, np.angle(baselineComplex15))
mag15 = np.insert(np.abs(data15['complex'].astype(complex)).values,0,np.abs(baselineComplex15))
amplitudes15 = np.insert(data15['stim_amp'],0,0)

ax.scatter(angle7,  mag7,  s=(amplitudes7*20),  color=blue)
ax.scatter(angle10, mag10, s=(amplitudes10*20), color=red)
ax.scatter(angle15, mag15, s=(amplitudes15*20), color=purple)

plot_condition_means(data7, baselineComplex7, ax, blue)
plot_condition_means(data10, baselineComplex10, ax, red)
plot_condition_means(data15,baselineComplex15,  ax, purple)

ax.get_figure().savefig("entrainmentPolar.png", dpi=300, bbox_inches="tight")

### Figure 3b

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,4))

ax.plot(np.insert(data7['stim_amp'],0,0),np.insert(data7['plv'],0,bplv7),'o',c=blue)
plot_condition_means_linear(data7,bplv7,ax,blue, label='7 Hz')

ax.plot(np.insert(data10['stim_amp'],0,0),np.insert(data10['plv'],0,bplv10),'o',c=red)
plot_condition_means_linear(data10,bplv10,ax,red, label='10 Hz')

ax.plot(np.insert(data15['stim_amp'],0,0),np.insert(data15['plv'],0,bplv15),'o',c=purple)
plot_condition_means_linear(data15,bplv15,ax,purple, label='15 Hz')

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('PLV')
plt.tight_layout()
plt.legend()
ax.get_figure().savefig("entrainmentPLV.png", dpi=300, bbox_inches="tight")

### Figure 3c

In [ ]:
stimFreqs = [7, 10, 15]

plt.figure(figsize=(4,4))

singleCellPhases = [-31,-36,-44]

bar_width = 0.25

    # plt.plot(allAmplitudes[i],allAngles[i]-singleCellPhases[i],c='C'+str(i),label=f'{stimFreqs[i]} Hz Stimulation')
plt.scatter(np.insert(data7['stim_amp'],0,0),angle7*180/np.pi-singleCellPhases[0],c=blue)
plt.scatter(np.insert(data10['stim_amp'],0,0),angle10*180/np.pi-singleCellPhases[1],c=red)
plt.scatter(np.insert(data15['stim_amp'],0,0),angle15*180/np.pi-singleCellPhases[2],c=purple)

plt.xlabel('Stimulus Amplitude (V/m)')
plt.ylabel('Difference between column phase delay\nand single-neuron phase delay (degrees)')
plt.legend()

plt.tight_layout()
plt.savefig('entrainmentAngles.png', dpi=600)
plt.show()

### Figure 3d (also Figure S2e)

In [ ]:
amps7 = np.insert(data7['stim_amp'],0,0)
baselinePyr7= getBaselinePower(baseline,7,'PYR')
baselineInh7 = getBaselinePower(baseline,7,'INT_INH')
baselineExc7 = getBaselinePower(baseline,7,'INT_EXC')

plvPyr7 = np.insert(data7['amplitude_norm_PYR'],0,baselinePyr7)
plvExc7 = np.insert(data7['amplitude_norm_INT_EXC'],0,baselineExc7)
plvInh7 = np.insert(data7['amplitude_norm_INT_INH'],0,baselineInh7)

fig, ax = plt.subplots(1,1,figsize=(4,4),sharey=True)

ax.scatter(amps7,plvPyr7,c='peru',label='Pyramidal')

ax.scatter(amps7,plvExc7,c='k',label='Exc. Interneuron')

ax.scatter(amps7,plvInh7,c=pink,label='Inh. Interneuron')

plot_condition_means_linear(data7,baselinePyr7,ax,'peru',id_col='stim_amp',z_col='plv_PYR')

plot_condition_means_linear(data7,baselineInh7,ax,pink,id_col='stim_amp',z_col='plv_INT_INH')

ax.legend()
ax.set_xlabel('Stimulus amplitude (V/m)')
ax.set_xlabel('Stimulus amplitude (V/m)')
plt.tight_layout()

### Figure S2 f-g

In [ ]:
amps10 = np.insert(data10['stim_amp'],0,0)
baselinePyr10 = getBaselinePower(baseline,10,'PYR')
baselineInh10 = getBaselinePower(baseline,10,'INT_INH')
baselineExc10 = getBaselinePower(baseline,10,'INT_EXC')

plvPyr10 = np.insert(data10['amplitude_norm_PYR'],0,baselinePyr10)
plvExc10 = np.insert(data10['amplitude_norm_INT_EXC'],0,baselineExc10)
plvInh10 = np.insert(data10['amplitude_norm_INT_INH'],0,baselineInh10)

fig, ax = plt.subplots(1,2,figsize=(8,4),sharey=True)

ax[0].scatter(amps10,plvPyr10,c='peru',label='Pyramidal')
ax[1].scatter(amps15,plvPyr15,c='peru',label='Pyramidal')
# plt.plot(np.array([np.sort(data['stim_amp'])[0],np.sort(data['stim_amp'])[-1]]),np.array([slope_pyr*np.sort(data['stim_amp'])[0]+intercept_pyr,np.sort(data['stim_amp'])[-1]*slope_pyr+intercept_pyr]),'maroon')


ax[0].scatter(amps10,plvExc10,c='k',label='Exc. Interneuron')
ax[1].scatter(amps15,plvExc15,c='k',label='Exc. Interneuron')

ax[0].scatter(amps10,plvInh10,c=pink,label='Inh. Interneuron')
ax[1].scatter(amps15,plvInh15,c=pink,label='Inh. Interneuron')
# plt.plot(np.array([np.sort(data['stim_amp'])[0],np.sort(data['stim_amp'])[-1]]),np.array([slope_inh*np.sort(data['stim_amp'])[0]+intercept_inh,np.sort(data['stim_amp'])[-1]*slope_inh+intercept_inh]),'royalblue')

plot_condition_means_linear(data10,baselinePyr10,ax[0],'peru',id_col='stim_amp',z_col='plv_PYR')
plot_condition_means_linear(data15,baselinePyr15,ax[1],'peru',id_col='stim_amp',z_col='plv_PYR')

plot_condition_means_linear(data10,baselineInh10,ax[0],pink,id_col='stim_amp',z_col='plv_INT_INH')
plot_condition_means_linear(data15,baselineInh15,ax[1],pink,id_col='stim_amp',z_col='plv_INT_INH')

ax[1].legend()
ax[0].set_xlabel('Stimulus amplitude (V/m)')
ax[1].set_xlabel('Stimulus amplitude (V/m)')
plt.tight_layout()

# plt.savefig('entrainmentCellType1015.png',dpi=300)

### The following cell just plots the legend for the previous figure

In [ ]:
amps10 = np.insert(data10['stim_amp'],0,0)
baselinePyr10 = getBaselinePLV(baseline,10,'PYR')
baselineInh10 = getBaselinePLV(baseline,10,'INT_INH')
baselineExc10 = getBaselinePLV(baseline,10,'INT_EXC')

plvPyr10 = np.insert(data10['plv_PYR'],0,baselinePyr10)
plvExc10 = np.insert(data10['plv_INT_EXC'],0,baselineExc10)
plvInh10 = np.insert(data10['plv_INT_INH'],0,baselineInh10)

amps15 = np.insert(data15['stim_amp'],0,0)
baselinePyr15 = getBaselinePLV(baseline,15,'PYR')
baselineInh15 = getBaselinePLV(baseline,15,'INT_INH')
baselineExc15 = getBaselinePLV(baseline,15,'INT_EXC')

plvPyr15 = np.insert(data15['plv_PYR'],0,baselinePyr15)
plvExc15 = np.insert(data15['plv_INT_EXC'],0,baselineExc15)
plvInh15 = np.insert(data15['plv_INT_INH'],0,baselineInh15)

fig, ax = plt.subplots(1,2,figsize=(8,4),sharey=True)

ax[0].scatter(amps10,plvPyr10,c='peru',label='Pyramidal')
ax[1].scatter(amps15,plvPyr15,c='peru',label='Pyramidal')
# plt.plot(np.array([np.sort(data['stim_amp'])[0],np.sort(data['stim_amp'])[-1]]),np.array([slope_pyr*np.sort(data['stim_amp'])[0]+intercept_pyr,np.sort(data['stim_amp'])[-1]*slope_pyr+intercept_pyr]),'maroon')


ax[0].scatter(amps10,plvExc10,c='k',label='Exc. Interneuron')
ax[1].scatter(amps15,plvExc15,c='k',label='Exc. Interneuron')

ax[0].scatter(amps10,plvInh10,c=pink,label='Inh. Interneuron')
ax[1].scatter(amps15,plvInh15,c=pink,label='Inh. Interneuron')
# plt.plot(np.array([np.sort(data['stim_amp'])[0],np.sort(data['stim_amp'])[-1]]),np.array([slope_inh*np.sort(data['stim_amp'])[0]+intercept_inh,np.sort(data['stim_amp'])[-1]*slope_inh+intercept_inh]),'royalblue')

plot_condition_means_linear(data10,baselinePyr10,ax[0],'peru',id_col='stim_amp',z_col='plv_PYR')
plot_condition_means_linear(data15,baselinePyr15,ax[1],'peru',id_col='stim_amp',z_col='plv_PYR')

plot_condition_means_linear(data10,baselineInh10,ax[0],pink,id_col='stim_amp',z_col='plv_INT_INH')
plot_condition_means_linear(data15,baselineInh15,ax[1],pink,id_col='stim_amp',z_col='plv_INT_INH')

ax[1].legend()
ax[0].set_xlabel('Stimulus amplitude (V/m)')
ax[1].set_xlabel('Stimulus amplitude (V/m)')
ax[0].set_ylabel('PLV')
plt.tight_layout()

plt.savefig('entrainmentCellType1015.png',dpi=300)

### Now we calculate some statistics on the phase preferences of individual cells

In [ ]:
def get_all_phases(amp,f):

    import bluepysnap as bp

    simsf = data[data['stim_freq'] == f]
    sims = simsf[simsf['stim_amp'] == a]

    for l in range(len(sims)):
        s = bp.Simulation(sims.iloc[l]['source_file'])
            
        c = s.circuit
            
        nodes = c.nodes['S1nonbarrel_neurons'].get()
            
        spikes = s.spikes['S1nonbarrel_neurons'].get()
        if l == 0:
            phases = get_phase(spikes, FREQ_HZ=f)
        else:
            phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))

    
    return phases

In [ ]:
amps = [4, 2, 1]

for a in amps:

    phase7 = get_all_phases(a, 7)
    phase15 = get_all_phases(a, 15)
    phase10 = get_all_phases(a, 10)
    
    print(kuiper_test(phase7,phase10, by_cell=True))
    print(kuiper_test(phase7['phase_rad'],phase10['phase_rad'], by_cell=False))
    print('\n')
    print(kuiper_test(phase10,phase15,by_cell=True))
    print(kuiper_test(phase10['phase_rad'],phase15['phase_rad']))
    print('\n\n')



## Figure 4

In [ ]:
amps = np.insert(simulations['stim_amp'],0,0)
baselinePyr = getBaselinePLV(baseline,7,'PYR')
baselineInh = getBaselinePLV(baseline,7,'INT_INH')
baselineExc = getBaselinePLV(baseline,7,'INT_EXC')

plvPyr = np.insert(data['plv_PYR'],0,baselinePyr)
plvExc = np.insert(data['plv_INT_EXC'],0,baselineExc)
plvInh = np.insert(data['plv_INT_INH'],0,baselineInh)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))

ax.scatter(amps,plvPyr,c='peru',label='Pyramidal')
# plt.plot(np.array([np.sort(data['stim_amp'])[0],np.sort(data['stim_amp'])[-1]]),np.array([slope_pyr*np.sort(data['stim_amp'])[0]+intercept_pyr,np.sort(data['stim_amp'])[-1]*slope_pyr+intercept_pyr]),'maroon')


ax.scatter(amps,plvExc,c='k',label='Exc. Interneuron')

ax.scatter(amps,plvInh,c=pink,label='Inh. Interneuron')

plot_condition_means_linear(data,baselinePyr,ax,'peru',id_col='stim_amp',z_col='plv_PYR')
plot_condition_means_linear(data,baselineInh,ax,pink,id_col='stim_amp',z_col='plv_INT_INH')
# plot_condition_means_linear(data,baselineExc,ax,'olive',id_col='stim_amp',z_col='plv_INT_EXC')

# plt.plot(np.array([np.sort(data['stim_amp'])[0],np.sort(data['stim_amp'])[-1]]),np.array([slope_inh*np.sort(data['stim_amp'])[0]+intercept_inh,np.sort(data['stim_amp'])[-1]*slope_inh+intercept_inh]),'royalblue')

# plt.legend(frameon=False)
plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('PLV')
plt.tight_layout()

plt.savefig('entrainmentCellType.png',dpi=300)

In [ ]:
from scipy import stats

slope, intercept, r_value, p_value, std_err = stats.linregress(data['stim_amp'], data['plv'])
r_squared = r_value**2

print(f"R^2 = {r_squared:.4f}")

In [ ]:
# Apparent nonlinearities in PLV disappear with additional seeds

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.plot(data['stim_amp'],data['plv'],'o')
ax.plot(np.array([np.sort(data['stim_amp'])[0],np.sort(data['stim_amp'])[-1]]),np.array([slope*np.sort(data['stim_amp'])[0]+intercept,np.sort(data['stim_amp'])[-1]*slope+intercept]),'k')
plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('PLV')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.plot(data['stim_amp'],data['amplitude_norm'],'o')
plt.tight_layout()

In [ ]:
from scipy import stats

slope, intercept, r_value, p_value, std_err = stats.linregress(data['stim_amp'], data['amplitude_norm'])
r_squared = r_value**2

print(f"R^2 = {r_squared:.4f}")

In [ ]:
np.squeeze(np.where(s==amps))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 8), subplot_kw={'projection': 'polar'},layout='constrained')

colorList = []
amps = np.sort(np.unique(data['stim_amp']))
for s in data['stim_amp']:
    color = np.squeeze(np.where(s==amps))
    colorList.append('C'+str(int(color)))

ax.scatter(np.angle(data['complex'].astype(complex)),np.abs(data['complex'].astype(complex)).values,color=colorList)


In [ ]:

colorList = []
amps = np.sort(np.unique(data['stim_amp']))
for s in data['stim_amp']:
    color = np.squeeze(np.where(s==amps))
    colorList.append('C'+str(int(color)))
    
fig, ax = plt.subplots(1, 1, figsize=(5, 8), subplot_kw={'projection': 'polar'},layout='constrained')
ax.scatter(np.angle(data['complex_INT_INH'].astype(complex)),np.abs(data['complex_INT_INH'].astype(complex)).values,color=colorList,marker='o')

fig, ax = plt.subplots(1, 1, figsize=(5, 8), subplot_kw={'projection': 'polar'},layout='constrained')
ax.scatter(np.angle(data['complex_INT_EXC'].astype(complex)),np.abs(data['complex_INT_EXC'].astype(complex)).values,color=colorList,marker='*')

fig, ax = plt.subplots(1, 1, figsize=(5, 8), subplot_kw={'projection': 'polar'},layout='constrained')
ax.scatter(np.angle(data['complex_PYR'].astype(complex)),np.abs(data['complex_PYR'].astype(complex)).values,color=colorList,marker='^')

In [ ]:
from scipy import stats

slope_inh, intercept_inh, r_value, p_value, std_err = stats.linregress(data['stim_amp'], data['plv_INT_INH'])
r_squared = r_value**2

print(f"R^2 = {r_squared:.4f}")

slope_pyr, intercept_pyr, r_value, p_value, std_err = stats.linregress(data['stim_amp'], data['plv_PYR'])
r_squared = r_value**2

print(f"R^2 = {r_squared:.4f}")

In [ ]:
from scipy import stats

slope, intercept, r_value, p_value, std_err = stats.linregress(data['stim_amp'], data['plv_INT_EXC'])
r_squared = r_value**2

print(f"R^2 = {r_squared:.4f}")

# Now we plot Figure 4a

In [ ]:
#Choose a simulation you want to look at
sim_id = 'highCorrelation_10Hz2Vm'
#choose bin size for firing rate calculation
bin = 5
#get the location of the config file for that simulation
config_path = simulations.loc[simulations['sim_id'] == sim_id, 'source_file'].values[0]
freq = simulations.loc[simulations['sim_id'] == sim_id, 'stim_freq'].values[0]
delay = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_start'].values[0]
time_stop = simulations.loc[simulations['sim_id'] == sim_id, 'analysis_stop'].values[0]
#load the data for that simulation
print(f"Loading data for simulation {sim_id} from {config_path}")
output_folder = os.path.join("output", sim_id)
os.makedirs(output_folder, exist_ok=True)

In [ ]:
simulationData, allData, cellData, spikeData = load_data(config_path)
cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

In [ ]:
psth_plot(spikeData, "all cells", cell_to_rank, simulationData, bin_width=bin, save_histogram=os.path.join(output_folder, f"psth_{sim_id}_{bin}bin_smoothed10.png"), smoothed=True,figsize=(12,6))

# Here we compare results from the different network regimes (Figure 4 b-e)

In [ ]:
data = simulations[simulations['sim_id'].str.contains('highCorrelation')]
data = data[~data['sim_id'].str.contains('baseline')]
data.sort_values('stim_amp',inplace=True)

data10 = data[data['stim_freq']==10]
data7 = data[data['stim_freq']==7]

orig = simulations[~simulations['sim_id'].str.contains('delay')]
orig = orig[~orig['sim_id'].str.contains('highCorrelation')]
orig = orig[~orig['sim_id'].str.contains('obi', case=False, na=False)]
orig.sort_values('stim_amp',inplace=True)

orig10 = orig[orig['stim_freq']==10]
orig7 = orig[orig['stim_freq']==7]

### Figure 4b

In [ ]:
# --- figure --------------------------------------------------------------
fig, ax = plt.subplots(1, 1, figsize=(4, 4),
                       subplot_kw={'projection': 'polar'},
                       layout='constrained')

# Externally-driven
baselineLow       = simulations[simulations['sim_id'] == 'baseline']
baselineComplex7  = getBaselineComplex(baselineLow, 7)
baselineComplex10 = getBaselineComplex(baselineLow, 10)
amplitudes7  = np.insert(orig7['stim_amp'],  0, 0)
amplitudes10 = np.insert(orig10['stim_amp'], 0, 0)

polar_scatter(ax, orig7,  baselineComplex7,  blue, 'Externally-driven, 7 Hz')
polar_scatter(ax, orig10, baselineComplex10, red, 'Externally-driven, 10 Hz')
plot_condition_means(orig7,  baselineComplex7,  ax, blue)
plot_condition_means(orig10, baselineComplex10, ax, red)

# Internally-driven
baselineHigh      = simulations[simulations['sim_id'] == 'baseline_highCorrelation_stopStart']
baselineComplex7  = getBaselineComplex(baselineHigh, 7)
baselineComplex10 = getBaselineComplex(baselineHigh, 10)
amplitudes7  = np.insert(data7['stim_amp'],  0, 0)
amplitudes10 = np.insert(data10['stim_amp'], 0, 0)

polar_scatter(ax, data7,  baselineComplex7,  lightblue,    'Internally-driven, 7 Hz')
polar_scatter(ax, data10, baselineComplex10, magenta, 'Internally-driven, 10 Hz')
plot_condition_means(data7,  baselineComplex7,  ax, 'lightblue')
plot_condition_means(data10, baselineComplex10, ax, magenta)

# --- legend with uniform marker size ------------------------------------
# leg = ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1.05), frameon=False)
# for handle in leg.legend_handles:
#     handle.set_sizes([40])

plt.savefig('brainstatePolar.png',dpi=600)

### Figure 4c

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,4))

baseline = simulations[simulations['sim_id']=='baseline']
bplv10 = getBaselinePLV(baseline,10)
bplv7 = getBaselinePLV(baseline,7)

ax.plot(np.insert(orig7['stim_amp'],0,0),np.insert(orig7['plv'],0,bplv7),'o',c=blue, label='Externally-driven, 7 Hz')
plot_condition_means_linear(orig7,bplv7,ax,blue, label='Internally-driven, 7 Hz')


ax.plot(np.insert(orig10['stim_amp'],0,0),np.insert(orig10['plv'],0,bplv10),'o',c=red, label='Externally-driven, 10 Hz')
plot_condition_means_linear(orig10,bplv10,ax,red, label='Internally-driven, 10 Hz')


baseline = simulations[simulations['sim_id']=='baseline_highCorrelation_stopStart']
bplv10 = getBaselinePLV(baseline,10)
bplv7 = getBaselinePLV(baseline,7)

ax.plot(np.insert(data7['stim_amp'],0,0),np.insert(data7['plv'],0,bplv7),'o',c=lightblue, label='Internally-driven, 7 Hz')
plot_condition_means_linear(data7,bplv7,ax,lightblue, label='Extrernally-driven, 7 Hz')


ax.plot(np.insert(data10['stim_amp'],0,0),np.insert(data10['plv'],0,bplv10),'o',c=magenta, label='Internally-driven, 10 Hz')
plot_condition_means_linear(data10,bplv10,ax,magenta, label='Externally-driven, 10 Hz')



plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('PLV')
plt.tight_layout()
# plt.legend()
ax.get_figure().savefig("brainStatePLV.png", dpi=300, bbox_inches="tight")

### Figure 4d (also Figure S3g)

In [ ]:
baselineorig = getBaselineComplex(baselineLow, 10)
baseline = getBaselineComplex(baselineHigh, 10)

fig, ax = plt.subplots(1, 1, figsize=(4,4), subplot_kw={'projection': 'polar'},layout='constrained')

polar_scatter(ax,data10,baseline,lighterpink,z_col='complex_INT_INH') 
plot_condition_means(data10,  baseline,  ax, lighterpink, z_col='complex_INT_INH')

polar_scatter(ax,orig10,baselineorig,pink,z_col='complex_INT_INH') 
plot_condition_means(orig10,  baselineorig,  ax, pink, z_col='complex_INT_INH')


polar_scatter(ax,data10,baseline,midgrey,z_col='complex_INT_EXC') 
plot_condition_means(data10,  baseline,  ax, midgrey, z_col='complex_INT_EXC')

polar_scatter(ax,orig10,baselineorig,'k',z_col='complex_INT_EXC') 
plot_condition_means(orig10,  baselineorig,  ax, 'k', z_col='complex_INT_EXC')


polar_scatter(ax,data10,baseline,lighterorange,z_col='complex_PYR') 
plot_condition_means(data10,  baseline,  ax, lighterorange, z_col='complex_PYR')

polar_scatter(ax,orig10,baselineorig,'peru',z_col='complex_PYR') 
plot_condition_means(orig10,  baselineorig,  ax, 'peru', z_col='complex_PYR')

plt.savefig('cellTypePolar_10Hz.png',dpi=600)


### Figure 4e (also Figure S3f and S3h)

In [ ]:
amps10 = np.insert(data10['stim_amp'],0,0)
baselinePyr10 = getBaselinePLV(baseline,10,'PYR')
baselineInh10 = getBaselinePLV(baseline,10,'INT_INH')
baselineExc10 = getBaselinePLV(baseline,10,'INT_EXC')

plvPyr10 = np.insert(data10['plv_PYR'],0,baselinePyr10)
plvExc10 = np.insert(data10['plv_INT_EXC'],0,baselineExc10)
plvInh10 = np.insert(data10['plv_INT_INH'],0,baselineInh10)

amps7 = np.insert(data7['stim_amp'],0,0)
baselinePyr7 = getBaselinePLV(baseline,7,'PYR')
baselineInh7 = getBaselinePLV(baseline,7,'INT_INH')
baselineExc7 = getBaselinePLV(baseline,7,'INT_EXC')

plvPyr7 = np.insert(data7['plv_PYR'],0,baselinePyr7)
plvExc7 = np.insert(data7['plv_INT_EXC'],0,baselineExc7)
plvInh7 = np.insert(data7['plv_INT_INH'],0,baselineInh7)

fig, ax = plt.subplots(1,2,figsize=(8,4),sharey=True)

ax[1].scatter(amps10,plvPyr10,c=lighterorange,label='Pyramidal')
ax[0].scatter(amps7,plvPyr7,c=lighterorange,label='Pyramidal')


ax[1].scatter(amps10,plvExc10,c=midgrey,label='Exc. Interneuron')
ax[0].scatter(amps7,plvExc7,c=midgrey,label='Exc. Interneuron')

ax[1].scatter(amps10,plvInh10,c=lighterpink,label='Inh. Interneuron')
ax[0].scatter(amps7,plvInh7,c=lighterpink,label='Inh. Interneuron')

plot_condition_means_linear(data10,baselinePyr10,ax[1],lighterorange,id_col='stim_amp',z_col='plv_PYR')
plot_condition_means_linear(data7,baselinePyr7,ax[0],lighterorange,id_col='stim_amp',z_col='plv_PYR')

plot_condition_means_linear(data10,baselineInh10,ax[1],lighterpink,id_col='stim_amp',z_col='plv_INT_INH')
plot_condition_means_linear(data7,baselineInh7,ax[0],lighterpink,id_col='stim_amp',z_col='plv_INT_INH')

ax[1].legend()
ax[0].set_xlabel('Stimulus amplitude (V/m)')
ax[1].set_xlabel('Stimulus amplitude (V/m)')
ax[0].set_ylabel('PLV')
plt.tight_layout()

plt.savefig('cellTypeHighCorrelation.png',dpi=600)

### Figure S3e

In [ ]:
baselineorig = getBaselineComplex(baselineLow, 7)
baseline = getBaselineComplex(baselineHigh, 7)

fig, ax = plt.subplots(1, 1, figsize=(4,4), subplot_kw={'projection': 'polar'},layout='constrained')

polar_scatter(ax,data7,baseline,lighterpink,z_col='complex_INT_INH') 
plot_condition_means(data7,  baseline,  ax, lighterpink, z_col='complex_INT_INH')

polar_scatter(ax,orig7,baselineorig,pink,z_col='complex_INT_INH',max_amp=2) 
plot_condition_means(orig7,  baselineorig,  ax, pink, z_col='complex_INT_INH',max_amp=2)


polar_scatter(ax,data7,baseline,midgrey,z_col='complex_INT_EXC') 
plot_condition_means(data7,  baseline,  ax, midgrey, z_col='complex_INT_EXC')

polar_scatter(ax,orig7,baselineorig,'k',z_col='complex_INT_EXC',max_amp=2) 
plot_condition_means(orig7,  baselineorig,  ax, 'k', z_col='complex_INT_EXC',max_amp=2)


polar_scatter(ax,data7,baseline,lighterorange,z_col='complex_PYR') 
plot_condition_means(data7,  baseline,  ax, lighterorange, z_col='complex_PYR')

polar_scatter(ax,orig7,baselineorig,'peru',z_col='complex_PYR',max_amp=2) 
plot_condition_means(orig7,  baselineorig,  ax, 'peru', z_col='complex_PYR',max_amp=2)

plt.savefig('cellTypePolar_7Hz.png',dpi=600)


# Here we look at phase preferences of individual neurons in different simulations

## First for low-correlation circuit: Figure S2 a-d

In [ ]:
import bluepysnap as bp

In [ ]:
data = simulations[~simulations['sim_id'].str.contains('delay')]
data = data[~data['sim_id'].str.contains('highCorrelation')]
data = data[~data['sim_id'].str.contains('obi', case=False, na=False)]
data = data[~data['sim_id'].str.contains('hex0', case=False, na=False)]
data = data[~data['sim_id'].str.contains('replay', case=False, na=False)]

data.sort_values(['stim_freq','stim_amp',],inplace=True)

### Figure S2a

In [ ]:
allPhases = [[[], [], [], [], [], []], [[], [], [], [], [], []], [[], [], [], [], [], []]]
freqs = [7, 10, 15]
colorList = [blue, red, purple]
amps = [0, 0.25, .5, 1, 2, 4]
CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded
fig, ax = plt.subplots(3, 6, subplot_kw={"projection": "polar"},
                       figsize=(6 * CELL, 3 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

# cache cell positions now — some axes get removed below
pos = [[ax[r][c].get_position() for c in range(len(amps))] for r in range(len(freqs))]

for row, f in enumerate(freqs):
    simsf = data[data['stim_freq'] == f]
    for col, a in enumerate(amps):
        if a == 0:
            sims = data[data['sim_id'] == 'baseline']
        else:
            sims = simsf[simsf['stim_amp'] == a]
        if len(sims) == 0:
            ax[row][col].remove()      # drop the empty panel entirely
            continue
        for l in range(len(sims)):
            s = bp.Simulation(sims.iloc[l]['source_file'])
            spikes = s.spikes['S1nonbarrel_neurons'].get()
            if l == 0:
                phases = get_phase(spikes, FREQ_HZ=f)
            else:
                phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))
        plot_rose(phases, ax=ax[row][col], color=colorList[row], arrow_scale=0.05)
        allPhases[row][col] = phases

# --- column titles (field amplitude) and row titles (reference frequency) ---
for col, a in enumerate(amps):
    label = "baseline" if a == 0 else f"{a:g} V/m"
    fig.text(0.5 * (pos[0][col].x0 + pos[0][col].x1), 0.955, label,
             ha="center", va="center", fontsize=9)
for row, f in enumerate(freqs):
    fig.text(0.015, 0.5 * (pos[row][0].y0 + pos[row][0].y1), f"{f} Hz",
             ha="center", va="center", fontsize=9, rotation=90,
             color=colorList[row])

fig.savefig("rose_grid.png",dpi=600, bbox_inches="tight", pad_inches=0)

### Statistical testing for differences between baseline and stimulated behavior at each frequency

In [ ]:
for i in np.arange(1,6):
    for j in range(3):
        try:
            print(kuiper_test(allPhases[j][0]['phase_rad'],allPhases[j][i]['phase_rad']))
        except:
            print(i)
    print('\n')

### Figure S2b

In [ ]:
allPhasesPyr = [ [[], [], [], [], [], []], [[], [], [], [], [], []], [[], [], [], [], [], []] ]
freqs = [7, 10, 15]
colorList = ['peru','peru', 'peru']
amps = [0, 0.25, .5, 1, 2, 4]
CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded
fig, ax = plt.subplots(3, 6, subplot_kw={"projection": "polar"},
                       figsize=(6 * CELL, 3 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

# cache cell positions now — some axes get removed below
pos = [[ax[r][c].get_position() for c in range(len(amps))] for r in range(len(freqs))]

for row, f in enumerate(freqs):
    simsf = data[data['stim_freq'] == f]
    for col, a in enumerate(amps):
        if a == 0:
            sims = data[data['sim_id'] == 'baseline']
        else:
            sims = simsf[simsf['stim_amp'] == a]
        if len(sims) == 0:
            ax[row][col].remove()      # drop the empty panel entirely
            continue
        for l in range(len(sims)):
            s = bp.Simulation(sims.iloc[l]['source_file'])
            
            c = s.circuit
            
            nodes = c.nodes['S1nonbarrel_neurons'].get()
            pyr = nodes[nodes['morph_class']=='PYR'].index
            intr = nodes[nodes['morph_class']=='INT']
            int_inh = intr[intr['synapse_class']=='INH'].index
            int_exc = intr[intr['synapse_class']=='EXC'].index
            
            spikes = s.spikes['S1nonbarrel_neurons'].get(pyr)
            if l == 0:
                phases = get_phase(spikes, FREQ_HZ=f)
            else:
                phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))
        plot_rose(phases, ax=ax[row][col], color=colorList[row], arrow_scale=0.05)
        allPhasesPyr[row][col] = phases

# --- column titles (field amplitude) and row titles (reference frequency) ---
for col, a in enumerate(amps):
    label = "baseline" if a == 0 else f"{a:g} V/m"
    fig.text(0.5 * (pos[0][col].x0 + pos[0][col].x1), 0.955, label,
             ha="center", va="center", fontsize=9)
for row, f in enumerate(freqs):
    fig.text(0.015, 0.5 * (pos[row][0].y0 + pos[row][0].y1), f"{f} Hz",
             ha="center", va="center", fontsize=9, rotation=90,
             color=colorList[row])

fig.savefig("rose_grid_pyr.png",dpi=600, bbox_inches="tight", pad_inches=0)

### Figure S2c

In [ ]:
allPhasesIntInh = [ [[], [], [], [], [], []], [[], [], [], [], [], []], [[], [], [], [], [], []] ]
freqs = [7, 10, 15]
colorList = [pink, pink, pink]
amps = [0, 0.25, .5, 1, 2, 4]
CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded
fig, ax = plt.subplots(3, 6, subplot_kw={"projection": "polar"},
                       figsize=(6 * CELL, 3 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

# cache cell positions now — some axes get removed below
pos = [[ax[r][c].get_position() for c in range(len(amps))] for r in range(len(freqs))]

for row, f in enumerate(freqs):
    simsf = data[data['stim_freq'] == f]
    for col, a in enumerate(amps):
        if a == 0:
            sims = data[data['sim_id'] == 'baseline']
        else:
            sims = simsf[simsf['stim_amp'] == a]
        if len(sims) == 0:
            ax[row][col].remove()      # drop the empty panel entirely
            continue
        for l in range(len(sims)):
            s = bp.Simulation(sims.iloc[l]['source_file'])
            
            c = s.circuit
            
            nodes = c.nodes['S1nonbarrel_neurons'].get()
            pyr = nodes[nodes['morph_class']=='PYR'].index
            intr = nodes[nodes['morph_class']=='INT']
            int_inh = intr[intr['synapse_class']=='INH'].index
            int_exc = intr[intr['synapse_class']=='EXC'].index
            
            spikes = s.spikes['S1nonbarrel_neurons'].get(int_inh)
            if l == 0:
                phases = get_phase(spikes, FREQ_HZ=f)
            else:
                phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))
        plot_rose(phases, ax=ax[row][col], color=colorList[row], arrow_scale=0.05)
        allPhasesIntInh[row][col] = phases

# --- column titles (field amplitude) and row titles (reference frequency) ---
for col, a in enumerate(amps):
    label = "baseline" if a == 0 else f"{a:g} V/m"
    fig.text(0.5 * (pos[0][col].x0 + pos[0][col].x1), 0.955, label,
             ha="center", va="center", fontsize=9)
for row, f in enumerate(freqs):
    fig.text(0.015, 0.5 * (pos[row][0].y0 + pos[row][0].y1), f"{f} Hz",
             ha="center", va="center", fontsize=9, rotation=90,
             color=colorList[row])

fig.savefig("rose_grid_intinh.png",dpi=600, bbox_inches="tight", pad_inches=0)

### Statistical test for differences in inhibitory behavior relative to baseline

In [ ]:
kuiper_test(allPhasesIntInh[0][0]['phase_rad'],allPhasesIntInh[0][5]['phase_rad'])

### Figure S2d

In [ ]:
allPhasesIntExc = [ [[], [], [], [], [], []], [[], [], [], [], [], []], [[], [], [], [], [], []] ]
freqs = [7, 10, 15]
colorList = ['k','k','k']
amps = [0, 0.25, .5, 1, 2, 4]
CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded
fig, ax = plt.subplots(3, 6, subplot_kw={"projection": "polar"},
                       figsize=(6 * CELL, 3 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

# cache cell positions now — some axes get removed below
pos = [[ax[r][c].get_position() for c in range(len(amps))] for r in range(len(freqs))]

for row, f in enumerate(freqs):
    simsf = data[data['stim_freq'] == f]
    for col, a in enumerate(amps):
        if a == 0:
            sims = data[data['sim_id'] == 'baseline']
        else:
            sims = simsf[simsf['stim_amp'] == a]
        if len(sims) == 0:
            ax[row][col].remove()      # drop the empty panel entirely
            continue
        for l in range(len(sims)):
            s = bp.Simulation(sims.iloc[l]['source_file'])
            
            c = s.circuit
            
            nodes = c.nodes['S1nonbarrel_neurons'].get()
            pyr = nodes[nodes['morph_class']=='PYR'].index
            intr = nodes[nodes['morph_class']=='INT']
            int_inh = intr[intr['synapse_class']=='INH'].index
            int_exc = intr[intr['synapse_class']=='EXC'].index
            
            spikes = s.spikes['S1nonbarrel_neurons'].get(int_exc)
            if l == 0:
                phases = get_phase(spikes, FREQ_HZ=f)
            else:
                phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))
        plot_rose(phases, ax=ax[row][col], color=colorList[row], arrow_scale=0.05)
        allPhasesIntExc[row][col] = phases

# --- column titles (field amplitude) and row titles (reference frequency) ---
for col, a in enumerate(amps):
    label = "baseline" if a == 0 else f"{a:g} V/m"
    fig.text(0.5 * (pos[0][col].x0 + pos[0][col].x1), 0.955, label,
             ha="center", va="center", fontsize=9)
for row, f in enumerate(freqs):
    fig.text(0.015, 0.5 * (pos[row][0].y0 + pos[row][0].y1), f"{f} Hz",
             ha="center", va="center", fontsize=9, rotation=90,
             color=colorList[row])

fig.savefig("rose_grid_intexc.png",dpi=600, bbox_inches="tight", pad_inches=0)

## Now for high-correlation circuit: Figure S3a-d

In [ ]:
data = simulations[~simulations['sim_id'].str.contains('delay')]
data = data[data['sim_id'].str.contains('highCorrelation')]
data = data[~data['sim_id'].str.contains('replay', case=False, na=False)]

data.sort_values(['stim_freq','stim_amp',],inplace=True)

### Figure S3a

In [ ]:
allPhasesHC = [[[], [], [], [], [],[]], [[], [], [], [], [],[]]]
freqs = [7, 10]
colorList = [lightblue,magenta]
amps = [0, 0.25, .5, 1, 2, 4]
CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded
fig, ax = plt.subplots(2, 6, subplot_kw={"projection": "polar"},
                       figsize=(6 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

# cache cell positions now — some axes get removed below
pos = [[ax[r][c].get_position() for c in range(len(amps))] for r in range(len(freqs))]

for row, f in enumerate(freqs):
    simsf = data[data['stim_freq'] == f]
    for col, a in enumerate(amps):
        if a == 0:
            sims = data[data['sim_id'] == 'baseline_highCorrelation_stopStart']
        else:
            sims = simsf[simsf['stim_amp'] == a]
        if len(sims) == 0:
            ax[row][col].remove()      # drop the empty panel entirely
            continue
        for l in range(len(sims)):
            s = bp.Simulation(sims.iloc[l]['source_file'])
            spikes = s.spikes['S1nonbarrel_neurons'].get()
            if l == 0:
                phases = get_phase(spikes, FREQ_HZ=f)
            else:
                phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))
        plot_rose(phases, ax=ax[row][col], color=colorList[row], arrow_scale=0.05)
        allPhasesHC[row][col] = phases

# --- column titles (field amplitude) and row titles (reference frequency) ---
for col, a in enumerate(amps):
    label = "baseline" if a == 0 else f"{a:g} V/m"
    fig.text(0.5 * (pos[0][col].x0 + pos[0][col].x1), 0.955, label,
             ha="center", va="center", fontsize=9)
for row, f in enumerate(freqs):
    fig.text(0.015, 0.5 * (pos[row][0].y0 + pos[row][0].y1), f"{f} Hz",
             ha="center", va="center", fontsize=9, rotation=90,
             color=colorList[row])

fig.savefig("rose_grid_hc.png",dpi=600, bbox_inches="tight", pad_inches=0)

### Statistical testing for differences from baseline

In [ ]:
for i in np.arange(1,6):
    for j in range(2):
        try:
            print(kuiper_test(allPhasesHC[j][0]['phase_rad'],allPhasesHC[j][i]['phase_rad']))
        except:
            print(i)
    print('\n')

### Statistical test for differences between low-calcium and high-calcium circuits

In [ ]:
for amp in [5,4,3,2,1]:
    print(kuiper_test(allPhasesHC[1][amp]['phase_rad'],allPhases[1][amp]['phase_rad']))

In [ ]:
for amp in [4,3,2,1]:
    print(kuiper_test(allPhasesHC[0][amp]['phase_rad'],allPhases[0][amp]['phase_rad']))

### Figure S2b

In [ ]:
allPhasesHCPyr = [ [[], [], [], [], [],[]], [[], [], [], [], [],[]] ]
freqs = [7, 10]
colorList = [lighterorange,lighterorange]
amps = [0, 0.25, .5, 1, 2, 4]
CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded
fig, ax = plt.subplots(2, 6, subplot_kw={"projection": "polar"},
                       figsize=(6 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

# cache cell positions now — some axes get removed below
pos = [[ax[r][c].get_position() for c in range(len(amps))] for r in range(len(freqs))]

for row, f in enumerate(freqs):
    simsf = data[data['stim_freq'] == f]
    for col, a in enumerate(amps):
        if a == 0:
            sims = data[data['sim_id'] == 'baseline_highCorrelation_stopStart']
        else:
            sims = simsf[simsf['stim_amp'] == a]
        if len(sims) == 0:
            ax[row][col].remove()      # drop the empty panel entirely
            continue
        for l in range(len(sims)):
            s = bp.Simulation(sims.iloc[l]['source_file'])

            c = s.circuit
            
            nodes = c.nodes['S1nonbarrel_neurons'].get()
            pyr = nodes[nodes['morph_class']=='PYR'].index
            intr = nodes[nodes['morph_class']=='INT']
            int_inh = intr[intr['synapse_class']=='INH'].index
            int_exc = intr[intr['synapse_class']=='EXC'].index
            
            spikes = s.spikes['S1nonbarrel_neurons'].get(pyr)
            if l == 0:
                phases = get_phase(spikes, FREQ_HZ=f)
            else:
                phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))
        plot_rose(phases, ax=ax[row][col], color=colorList[row], arrow_scale=0.05)
        allPhasesHCPyr[row][col] = phases

# --- column titles (field amplitude) and row titles (reference frequency) ---
for col, a in enumerate(amps):
    label = "baseline" if a == 0 else f"{a:g} V/m"
    fig.text(0.5 * (pos[0][col].x0 + pos[0][col].x1), 0.955, label,
             ha="center", va="center", fontsize=9)
for row, f in enumerate(freqs):
    fig.text(0.015, 0.5 * (pos[row][0].y0 + pos[row][0].y1), f"{f} Hz",
             ha="center", va="center", fontsize=9, rotation=90,
             color=colorList[row])

fig.savefig("rose_grid_hcPyr.png",dpi=600, bbox_inches="tight", pad_inches=0)

### Figure S2c

In [ ]:
allPhasesHCInh = [ [[], [], [], [], [],[]], [[], [], [], [], [],[]] ]
freqs = [7, 10]
colorList = [lighterpink,lighterpink]
amps = [0, 0.25, .5, 1, 2, 4]
CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded
fig, ax = plt.subplots(2, 6, subplot_kw={"projection": "polar"},
                       figsize=(6 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

# cache cell positions now — some axes get removed below
pos = [[ax[r][c].get_position() for c in range(len(amps))] for r in range(len(freqs))]

for row, f in enumerate(freqs):
    simsf = data[data['stim_freq'] == f]
    for col, a in enumerate(amps):
        if a == 0:
            sims = data[data['sim_id'] == 'baseline_highCorrelation_stopStart']
        else:
            sims = simsf[simsf['stim_amp'] == a]
        if len(sims) == 0:
            ax[row][col].remove()      # drop the empty panel entirely
            continue
        for l in range(len(sims)):
            s = bp.Simulation(sims.iloc[l]['source_file'])

            c = s.circuit
            
            nodes = c.nodes['S1nonbarrel_neurons'].get()
            pyr = nodes[nodes['morph_class']=='PYR'].index
            intr = nodes[nodes['morph_class']=='INT']
            int_inh = intr[intr['synapse_class']=='INH'].index
            int_exc = intr[intr['synapse_class']=='EXC'].index
            
            spikes = s.spikes['S1nonbarrel_neurons'].get(int_inh)
            if l == 0:
                phases = get_phase(spikes, FREQ_HZ=f)
            else:
                phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))
        plot_rose(phases, ax=ax[row][col], color=colorList[row], arrow_scale=0.05)
        allPhasesHCInh[row][col] = phases

# --- column titles (field amplitude) and row titles (reference frequency) ---
for col, a in enumerate(amps):
    label = "baseline" if a == 0 else f"{a:g} V/m"
    fig.text(0.5 * (pos[0][col].x0 + pos[0][col].x1), 0.955, label,
             ha="center", va="center", fontsize=9)
for row, f in enumerate(freqs):
    fig.text(0.015, 0.5 * (pos[row][0].y0 + pos[row][0].y1), f"{f} Hz",
             ha="center", va="center", fontsize=9, rotation=90,
             color=colorList[row])

fig.savefig("rose_grid_hcInh.png",dpi=600, bbox_inches="tight", pad_inches=0)

### Statistical tests for differences between inhibitory cell behavior when stimulated relative to baseline

In [ ]:
for amp in [5, 4,3,2,1]:
    print(kuiper_test(allPhasesHCInh[1][0]['phase_rad'],allPhasesHCInh[1][amp]['phase_rad']))


In [ ]:
for amp in [4,3,2,1]:
    print(kuiper_test(allPhasesHCInh[0][0]['phase_rad'],allPhasesHCInh[0][amp]['phase_rad']))

### Figure S2d

In [ ]:
allPhasesHCIntExc = [ [[], [], [], [], [],[]], [[], [], [], [], [],[]] ]
freqs = [7, 10]
colorList = [midgrey,midgrey]
amps = [0, 0.25, .5, 1, 2, 4]
CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded
fig, ax = plt.subplots(2, 6, subplot_kw={"projection": "polar"},
                       figsize=(6 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

# cache cell positions now — some axes get removed below
pos = [[ax[r][c].get_position() for c in range(len(amps))] for r in range(len(freqs))]

for row, f in enumerate(freqs):
    simsf = data[data['stim_freq'] == f]
    for col, a in enumerate(amps):
        if a == 0:
            sims = data[data['sim_id'] == 'baseline_highCorrelation_stopStart']
        else:
            sims = simsf[simsf['stim_amp'] == a]
        if len(sims) == 0:
            ax[row][col].remove()      # drop the empty panel entirely
            continue
        for l in range(len(sims)):
            s = bp.Simulation(sims.iloc[l]['source_file'])

            c = s.circuit
            
            nodes = c.nodes['S1nonbarrel_neurons'].get()
            pyr = nodes[nodes['morph_class']=='PYR'].index
            intr = nodes[nodes['morph_class']=='INT']
            int_inh = intr[intr['synapse_class']=='INH'].index
            int_exc = intr[intr['synapse_class']=='EXC'].index
            
            spikes = s.spikes['S1nonbarrel_neurons'].get(int_exc)
            if l == 0:
                phases = get_phase(spikes, FREQ_HZ=f)
            else:
                phases = pd.concat((phases, get_phase(spikes, FREQ_HZ=f)))
        plot_rose(phases, ax=ax[row][col], color=colorList[row], arrow_scale=0.05)
        allPhasesHCPyr[row][col] = phases

# --- column titles (field amplitude) and row titles (reference frequency) ---
for col, a in enumerate(amps):
    label = "baseline" if a == 0 else f"{a:g} V/m"
    fig.text(0.5 * (pos[0][col].x0 + pos[0][col].x1), 0.955, label,
             ha="center", va="center", fontsize=9)
for row, f in enumerate(freqs):
    fig.text(0.015, 0.5 * (pos[row][0].y0 + pos[row][0].y1), f"{f} Hz",
             ha="center", va="center", fontsize=9, rotation=90,
             color=colorList[row])

fig.savefig("rose_grid_hcIntExc.png",dpi=600, bbox_inches="tight", pad_inches=0)

### Tests if the dip in PLV observed at 1 V/m in 10Hz internally-driven circuit is real 

In [ ]:
kuiper_test(allPhasesHC[1][1]['phase_rad'],allPhasesHC[1][3]['phase_rad'])

# Now we look at the replay simulations

In [ ]:
replay_low = simulations[simulations['sim_id']=='replay_10Hz2Vm']
connectedLow = simulations[simulations['sim_id']=='10Hz2Vm']

replay_high = simulations[simulations['sim_id']=='replay_10Hz2Vm_highCorrelation']
connectedHigh = simulations[simulations['sim_id']=='highCorrelation_10Hz2Vm']

### Figure 5a.i

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,4), subplot_kw={'projection': 'polar'},layout='constrained')

polar_scatter(ax,connectedLow,baseline_complex=None,color=red) 
polar_scatter(ax,replay_low,baseline_complex=None,color='maroon') 

polar_scatter(ax,connectedHigh,baseline_complex=None,color=magenta) 
polar_scatter(ax,replay_high,baseline_complex=None,color='indigo') 

plt.savefig('replayPolar.png',dpi=300)

### Figure 5b.i

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,4), subplot_kw={'projection': 'polar'},layout='constrained')

polar_scatter(ax,connectedLow,baseline_complex=None,color='peru',z_col='complex_PYR') 
polar_scatter(ax,replay_low,baseline_complex=None,color='peru',z_col='complex_PYR',marker='x') 

polar_scatter(ax,connectedLow,baseline_complex=None,color=pink,z_col='complex_INT_INH') 
polar_scatter(ax,replay_low,baseline_complex=None,color=pink,z_col='complex_INT_INH',marker='x') 

polar_scatter(ax,connectedLow,baseline_complex=None,color='k',z_col='complex_INT_EXC') 
polar_scatter(ax,replay_low,baseline_complex=None,color='k',z_col='complex_INT_EXC',marker='x') 

plt.savefig('replayPolar_cellTypeLow.png',dpi=300)

### Figure 5c.i

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,4), subplot_kw={'projection': 'polar'},layout='constrained')

polar_scatter(ax,connectedHigh,baseline_complex=None,color=lighterorange,z_col='complex_PYR') 
polar_scatter(ax,replay_high,baseline_complex=None,color=lighterorange,z_col='complex_PYR',marker='x') 

polar_scatter(ax,connectedHigh,baseline_complex=None,color=lighterpink,z_col='complex_INT_INH') 
polar_scatter(ax,replay_high,baseline_complex=None,color=lighterpink,z_col='complex_INT_INH',marker='x') 

polar_scatter(ax,connectedHigh,baseline_complex=None,color=midgrey,z_col='complex_INT_EXC') 
polar_scatter(ax,replay_high,baseline_complex=None,color=midgrey,z_col='complex_INT_EXC',marker='x') 

plt.savefig('replayPolar_cellTypeHigh.png',dpi=300)

### Figure 5a.ii

In [ ]:
import bluepysnap as bp

CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded

fig, ax = plt.subplots(2, 2, subplot_kw={"projection": "polar"},
                       figsize=(2 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

s = bp.Simulation(connectedLow['source_file'].iloc[0])
c = s.circuit
nodes = c.nodes['S1nonbarrel_neurons'].get()
spikes = s.spikes['S1nonbarrel_neurons'].get()

phases = get_phase(spikes, FREQ_HZ=10)

plot_rose(phases, ax=ax[0][0], color=red, arrow_scale=0.05)

s = bp.Simulation(replay_low['source_file'].iloc[0])
c = s.circuit
nodes = c.nodes['S1nonbarrel_neurons'].get()
spikes = s.spikes['S1nonbarrel_neurons'].get()

phases = get_phase(spikes, FREQ_HZ=10)

plot_rose(phases, ax=ax[1][0], color='maroon', arrow_scale=0.05)

s = bp.Simulation(connectedHigh['source_file'].iloc[0])
c = s.circuit
nodes = c.nodes['S1nonbarrel_neurons'].get()
spikes = s.spikes['S1nonbarrel_neurons'].get()

phases = get_phase(spikes, FREQ_HZ=10)

plot_rose(phases, ax=ax[0][1], color=magenta, arrow_scale=0.05)

s = bp.Simulation(replay_high['source_file'].iloc[0])
c = s.circuit
nodes = c.nodes['S1nonbarrel_neurons'].get()
spikes = s.spikes['S1nonbarrel_neurons'].get()

phases = get_phase(spikes, FREQ_HZ=10)

plot_rose(phases, ax=ax[1][1], color='indigo', arrow_scale=0.05)

plt.savefig('roseplotsReplay.png',dpi=600)

### Figure 5b.ii

In [ ]:
import bluepysnap as bp

CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded

fig, ax = plt.subplots(2, 3, subplot_kw={"projection": "polar"},
                       figsize=(3 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

s = bp.Simulation(connectedLow['source_file'].iloc[0])
c = s.circuit
nodes = c.nodes['S1nonbarrel_neurons'].get()

pyr = nodes[nodes['morph_class']=='PYR'].index
intr = nodes[nodes['morph_class']=='INT']
int_inh = intr[intr['synapse_class']=='INH'].index
int_exc = intr[intr['synapse_class']=='EXC'].index

spikes = s.spikes['S1nonbarrel_neurons'].get(pyr)
phases = get_phase(spikes, FREQ_HZ=10)
plot_rose(phases, ax=ax[0][0], color='peru', arrow_scale=0.05)

spikes = s.spikes['S1nonbarrel_neurons'].get(int_inh)
phases = get_phase(spikes, FREQ_HZ=10)
plot_rose(phases, ax=ax[0][1], color=pink, arrow_scale=0.05)

spikes = s.spikes['S1nonbarrel_neurons'].get(int_exc)
phases = get_phase(spikes, FREQ_HZ=10)
plot_rose(phases, ax=ax[0][2], color='k', arrow_scale=0.05)

s = bp.Simulation(replay_low['source_file'].iloc[0])
c = s.circuit
nodes = c.nodes['S1nonbarrel_neurons'].get()

pyr = nodes[nodes['morph_class']=='PYR'].index
intr = nodes[nodes['morph_class']=='INT']
int_inh = intr[intr['synapse_class']=='INH'].index
int_exc = intr[intr['synapse_class']=='EXC'].index

spikes = s.spikes['S1nonbarrel_neurons'].get(pyr)
phases = get_phase(spikes, FREQ_HZ=10)
plot_rose(phases, ax=ax[1][0], color='peru', arrow_scale=0.05)

spikes = s.spikes['S1nonbarrel_neurons'].get(int_inh)
phases = get_phase(spikes, FREQ_HZ=10)
plot_rose(phases, ax=ax[1][1], color=pink, arrow_scale=0.05)

spikes = s.spikes['S1nonbarrel_neurons'].get(int_exc)
phases = get_phase(spikes, FREQ_HZ=10)
plot_rose(phases, ax=ax[1][2], color='k', arrow_scale=0.05)

plt.savefig('roseplots_ReplayLow_CellTypes.png',dpi=600)

### Figure 5c.ii

In [ ]:
import bluepysnap as bp

CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded

fig, ax = plt.subplots(2, 3, subplot_kw={"projection": "polar"},
                       figsize=(3 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

s = bp.Simulation(connectedHigh['source_file'].iloc[0])
c = s.circuit
nodes = c.nodes['S1nonbarrel_neurons'].get()

pyr = nodes[nodes['morph_class']=='PYR'].index
intr = nodes[nodes['morph_class']=='INT']
int_inh = intr[intr['synapse_class']=='INH'].index
int_exc = intr[intr['synapse_class']=='EXC'].index

spikes = s.spikes['S1nonbarrel_neurons'].get(pyr)
phases = get_phase(spikes, FREQ_HZ=10)
phasesPyrConnected = phases
plot_rose(phases, ax=ax[0][0], color=lighterorange, arrow_scale=0.05)

spikes = s.spikes['S1nonbarrel_neurons'].get(int_inh)
phases = get_phase(spikes, FREQ_HZ=10)
phasesInhConnected = phases
plot_rose(phases, ax=ax[0][1], color=lighterpink, arrow_scale=0.05)

spikes = s.spikes['S1nonbarrel_neurons'].get(int_exc)
phases = get_phase(spikes, FREQ_HZ=10)
plot_rose(phases, ax=ax[0][2], color=midgrey, arrow_scale=0.05)

s = bp.Simulation(replay_high['source_file'].iloc[0])
c = s.circuit
nodes = c.nodes['S1nonbarrel_neurons'].get()

pyr = nodes[nodes['morph_class']=='PYR'].index
intr = nodes[nodes['morph_class']=='INT']
int_inh = intr[intr['synapse_class']=='INH'].index
int_exc = intr[intr['synapse_class']=='EXC'].index

spikes = s.spikes['S1nonbarrel_neurons'].get(pyr)
phases = get_phase(spikes, FREQ_HZ=10)
phasesPyrDisconneced = phases
plot_rose(phases, ax=ax[1][0], color=lighterorange, arrow_scale=0.05)

spikes = s.spikes['S1nonbarrel_neurons'].get(int_inh)
phases = get_phase(spikes, FREQ_HZ=10)
phasesInhDisconnected = phases
plot_rose(phases, ax=ax[1][1], color=lighterpink, arrow_scale=0.05)

spikes = s.spikes['S1nonbarrel_neurons'].get(int_exc)
phases = get_phase(spikes, FREQ_HZ=10)
plot_rose(phases, ax=ax[1][2], color=midgrey, arrow_scale=0.05)

plt.savefig('roseplots_ReplayHigh_CellTypes.png',dpi=600)

### Statistical test for differences between disconnected and connected circuit (high correlation)

In [ ]:
print(kuiper_test(phasesPyrConnected['phase_rad'],phasesPyrDisconneced['phase_rad']))
print(kuiper_test(phasesInhConnected['phase_rad'],phasesInhDisconnected['phase_rad']))

### Figure S4

In [ ]:
import bluepysnap as bp

CELL = 1.4  # inches per panel; square cells keep the polar circles from being padded

fig, ax = plt.subplots(2, 5, subplot_kw={"projection": "polar"},
                       figsize=(5 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
fig.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

figInh, axInh = plt.subplots(2, 5, subplot_kw={"projection": "polar"},
                       figsize=(5 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
figInh.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

figExc, axExc = plt.subplots(2, 5, subplot_kw={"projection": "polar"},
                       figsize=(5 * CELL, 2 * CELL))   # 6 columns, not 5
# leave a strip on the left and top for the labels; panels stay flush otherwise
figExc.subplots_adjust(left=0.05, right=1, bottom=0, top=0.94, wspace=0, hspace=0)

layers = [1,2,4,5,6]


for i, l in enumerate(layers):
    
    s = bp.Simulation(connectedHigh['source_file'].iloc[0])
    c = s.circuit
    nodes = c.nodes['S1nonbarrel_neurons'].get()

    if l == 2:
        lNodes = nodes[nodes['layer'].astype(int).isin([2, 3])]
    else:
        lNodes = nodes[nodes['layer'].astype(int)==l]

    pyr = lNodes[lNodes['morph_class']=='PYR'].index
    intr = lNodes[lNodes['morph_class']=='INT']
    int_inh = intr[intr['synapse_class']=='INH'].index
    int_exc = intr[intr['synapse_class']=='EXC'].index

    spikes = s.spikes['S1nonbarrel_neurons'].get(pyr)
    phases = get_phase(spikes, FREQ_HZ=10)
    plot_rose(phases, ax=ax[0][i], color=lighterorange, arrow_scale=0.05,normalize=False)
    
    spikes = s.spikes['S1nonbarrel_neurons'].get(int_inh)
    phases = get_phase(spikes, FREQ_HZ=10)
    plot_rose(phases, ax=axInh[0][i], color=lighterpink, arrow_scale=0.05,normalize=False)
    
    spikes = s.spikes['S1nonbarrel_neurons'].get(int_exc)
    phases = get_phase(spikes, FREQ_HZ=10)
    plot_rose(phases, ax=axExc[0][i], color=midgrey, arrow_scale=0.05,normalize=False)

    s = bp.Simulation(replay_high['source_file'].iloc[0])
    c = s.circuit
    nodes = c.nodes['S1nonbarrel_neurons'].get()

    if l == 2:
        lNodes = nodes[nodes['layer'].astype(int).isin([2, 3])]
    else:
        lNodes = nodes[nodes['layer'].astype(int)==l]

    pyr = lNodes[lNodes['morph_class']=='PYR'].index
    intr = lNodes[lNodes['morph_class']=='INT']
    int_inh = intr[intr['synapse_class']=='INH'].index
    int_exc = intr[intr['synapse_class']=='EXC'].index

    spikes = s.spikes['S1nonbarrel_neurons'].get(pyr)
    phases = get_phase(spikes, FREQ_HZ=10)
    plot_rose(phases, ax=ax[1][i], color=lighterorange, arrow_scale=0.05,normalize=False)
    
    spikes = s.spikes['S1nonbarrel_neurons'].get(int_inh)
    phases = get_phase(spikes, FREQ_HZ=10)
    plot_rose(phases, ax=axInh[1][i], color=lighterpink, arrow_scale=0.05,normalize=False)
    
    spikes = s.spikes['S1nonbarrel_neurons'].get(int_exc)
    phases = get_phase(spikes, FREQ_HZ=10)
    plot_rose(phases, ax=axExc[1][i], color=midgrey, arrow_scale=0.05,normalize=False)

rtop = sync_rose_scales([ax[0][0],ax[0][1],ax[0][2],ax[0][3],ax[0][4],ax[1][0],ax[1][1],ax[1][2],ax[1][3],ax[1][4]])
for row in ax:
    for col in row:
        col.set_yticks([rtop / 2, rtop])

rtop = sync_rose_scales([axInh[0][0],axInh[0][1],axInh[0][2],axInh[0][3],axInh[0][4],axInh[1][0],axInh[1][1],axInh[1][2],axInh[1][3],axInh[1][4]])
for row in axInh:
    for col in row:
        col.set_yticks([rtop / 2, rtop])
        
fig.savefig('roseplots_ReplayHigh_layersPyr.png',dpi=600)
figInh.savefig('roseplots_ReplayHigh_layersInh.png',dpi=600)


# Analyzes connectivity
### Makes Table S1

In [ ]:
import conntility

In [ ]:
fn_circ = '/mnt/e/O1_data/circuit_config.json'
import bluepysnap as bp
circ = bp.Circuit(fn_circ)


In [ ]:
from conntility import ConnectivityMatrix

load_cfg = {
    "loading": {
        "properties": ["x", "y", "z", "mtype", "etype", "layer", "synapse_class"],
        "base_target": "hex0",
    }
}

M = ConnectivityMatrix.from_bluepy(
    circ,                       # bluepysnap.Circuit on your circuit_config.json
    load_cfg,
    connectome="S1nonbarrel_neurons__S1nonbarrel_neurons__chemical",
    edge_property="delay",
    agg_func="mean",            # or "min"
)
print(M.edge_properties)        # confirm 'delay' made it in

In [ ]:
sc  = M.vertices["synapse_class"].values
lay = M.vertices["layer"].values.astype(str)

df = pd.DataFrame({
    "pre_layer":  lay[pre],
    "pre_sc":     sc[pre],
    "post_layer": lay[post],
    "post_sc":    sc[post],
    "delay":      M.edges.values[:,0],
})

tbl = df.groupby(["pre_layer", "pre_sc", "post_layer", "post_sc"])["delay"] \
        .agg(["mean", "std", "count"])

In [ ]:
pre_lbl  = pd.Series(lay[pre]).str.cat(pd.Series(sc[pre]),  sep="_")
post_lbl = pd.Series(lay[post]).str.cat(pd.Series(sc[post]), sep="_")

df = pd.DataFrame({"pre": pre_lbl, "post": post_lbl,
                   "delay": M.edges.values[:,0]})

mat = df.pivot_table(index="pre", columns="post", values="delay", aggfunc="mean")

In [ ]:
mat

# Below follow the plots for the 2026 posters: Delete these at some point

In [ ]:
stimFreqs = [7,10,15]

plt.figure(figsize=(8,8))

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id'] and 'delay' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']
    power = data['amplitude_norm']

    # amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
    # power = pd.concat([pd.Series(getBaselinePower(baseline,freq)),power],ignore_index=True)

    plt.plot(amplitudes,power,'-',label=f'{freq} Hz')
    plt.scatter(amplitudes,power,s=amplitudes*20,c='C'+str(j))

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('Normalized power at target frequency')
plt.legend()
plt.tight_layout()

In [ ]:
stimFreqs = [7,10,15]

plt.figure(figsize=(8,8))

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id'] and 'delay' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']
    power = data['power_norm']

    amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
    power = pd.concat([pd.Series(getBaselinePower(baseline,freq)),power],ignore_index=True)

    plt.plot(amplitudes,power,'-',label=f'{freq} Hz')
    plt.scatter(amplitudes,power,s=amplitudes*20,c='C'+str(j))

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('Normalized power at target frequency')
plt.legend()
plt.tight_layout()

plt.savefig('output/power.png',dpi=600)

In [ ]:
amplitudes

In [ ]:
stimFreqs = [7,10,15]

allDataReal = np.real(simulations['complex'].astype(complex))
allDataImag = np.imag(simulations['complex'].astype(complex))

maxReal = np.max(np.abs(allDataReal))
maxImag = np.max(np.abs(allDataImag))

plotLim = 8

plt.figure(figsize=(8,8))
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id'] and 'delay' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']

    realParts = np.real(data['complex'].astype(complex))
    imagParts = np.imag(data['complex'].astype(complex))

    baselineComplex = getBaselineComplex(baseline,freq)

    realParts = np.insert(realParts,0,np.real(baselineComplex))
    imagParts = np.insert(imagParts,0,np.imag(baselineComplex))
    amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)

    plt.plot(realParts,imagParts,'-',c='C'+str(j),label=f'{freq} Hz')
    plt.scatter(realParts,imagParts,s=(amplitudes+.01)*20,c='C'+str(j))

plt.xlim([-2,plotLim])
plt.ylim([-plotLim,2])

data = simulations[simulations['stim_freq']==7]
amplitudes = data['stim_amp']
amplitudes = np.insert(amplitudes,0,0)
freq_legend = plt.legend()


size_handles = [
    plt.scatter([], [], s=(a+.01) * 20, color='grey')
    for a in amplitudes
]

size_legend = plt.legend(
    size_handles,
    [f'{a}' for a in amplitudes],
    title='Stimulus amplitude (V/m)',
    loc='lower left'
)

plt.gca().add_artist(size_legend)

plt.xlabel('Real part')
plt.ylabel('Imaginary part')
plt.legend()
plt.tight_layout()
plt.savefig('output/complex.png',dpi=600)

In [ ]:
stimFreqs = [7,10,15]
delays = [0,25,50]
markers = ['-',':','--']

allDataReal = np.real(simulations['complex'].astype(complex))
allDataImag = np.imag(simulations['complex'].astype(complex))

maxReal = np.max(np.abs(allDataReal))
maxImag = np.max(np.abs(allDataImag))

plotLim = 8

plt.figure(figsize=(8,8))
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)

for j, freq in enumerate(stimFreqs):
    alldata = simulations[simulations['stim_freq']==freq]

    for k, delay in enumerate(delays):

        goodIdx = []
        for i in range(len(alldata)):
            if 'highCorrelation' not in alldata.iloc[i]['sim_id']:
                if delay == 0 and 'delay' not in alldata.iloc[i]['sim_id']:
                    print(alldata.iloc[i]['sim_id'])
                    goodIdx.append(i)
                elif delay != 0 and 'delay'+str(delay) in alldata.iloc[i]['sim_id']:
                    print(alldata.iloc[i]['sim_id'])
                    goodIdx.append(i)
        
        data = alldata.iloc[goodIdx]
    
        amplitudes = data['stim_amp']
    
        realParts = np.real(data['complex'].astype(complex))
        imagParts = np.imag(data['complex'].astype(complex))
    
        baselineComplex = getBaselineComplex(baseline,freq,cellType=None,delay=1500+delay)
    
        realParts = np.insert(realParts,0,np.real(baselineComplex))
        imagParts = np.insert(imagParts,0,np.imag(baselineComplex))
        amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
    
        plt.plot(realParts,imagParts,markers[k],c='C'+str(j),label=f'{freq} Hz')
        plt.scatter(realParts,imagParts,s=(amplitudes+.01)*20,c='C'+str(j))

        

# plt.xlim([-2,plotLim])
# plt.ylim([-plotLim,2])

# data = simulations[simulations['stim_freq']==7]
# amplitudes = data['stim_amp']
# amplitudes = np.insert(amplitudes,0,0)
# freq_legend = plt.legend()


# size_handles = [
#     plt.scatter([], [], s=(a+.01) * 20, color='grey')
#     for a in amplitudes
# ]

# size_legend = plt.legend(
#     size_handles,
#     [f'{a}' for a in amplitudes],
#     title='Stimulus amplitude (V/m)',
#     loc='lower left'
# )

# plt.gca().add_artist(size_legend)

plt.xlabel('Real part')
plt.ylabel('Imaginary part')
plt.legend()
plt.tight_layout()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

stimFreqs = [7, 10, 15]
delays = [0, 25, 50]
markers = ['-', ':', '--']

allDataReal = np.real(simulations['complex'].astype(complex))
allDataImag = np.imag(simulations['complex'].astype(complex))
maxReal = np.max(np.abs(allDataReal))
maxImag = np.max(np.abs(allDataImag))

# Create polar plot
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
ax.grid(True)

for j, freq in enumerate(stimFreqs):
    alldata = simulations[simulations['stim_freq'] == freq]
    for k, delay in enumerate(delays):
        goodIdx = []
        for i in range(len(alldata)):
            if 'highCorrelation' not in alldata.iloc[i]['sim_id']:
                if delay == 0 and 'delay' not in alldata.iloc[i]['sim_id']:
                    goodIdx.append(i)
                elif delay != 0 and 'delay' + str(delay) in alldata.iloc[i]['sim_id']:
                    goodIdx.append(i)
        
        data = alldata.iloc[goodIdx]
        
        amplitudes = data['stim_amp']
        
        realParts = np.real(data['complex'].astype(complex))
        imagParts = np.imag(data['complex'].astype(complex))
        
        baselineComplex = getBaselineComplex(baseline, freq, cellType=None, delay=1500 + delay)
        
        realParts = np.insert(realParts, 0, np.real(baselineComplex))
        imagParts = np.insert(imagParts, 0, np.imag(baselineComplex))
        amplitudes = pd.concat([pd.Series(0), amplitudes], ignore_index=True)
        
        # Convert to polar coordinates
        # radius: magnitude of complex number
        # theta: angle of complex number
        radii = np.sqrt(realParts**2 + imagParts**2)
        thetas = np.arctan2(imagParts, realParts)

        print(freq)
        print(thetas[0]*180/np.pi)
        
        # Plot in polar coordinates
        ax.plot(thetas, radii, markers[k], c='C' + str(j), label=f'{freq} Hz')
        ax.scatter(thetas, radii, s=(amplitudes + 0.01) * 20, c='C' + str(j))

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
# ax.set_rlim(0, 2)
plt.show()

In [ ]:
360 / (1/15 * 1e3) * 25

In [ ]:
delay = 1550
true_delay = None
t0 = delay - true_delay if true_delay is not None else 0.0
print(t0)

In [ ]:
k

In [ ]:
stimFreqs = [7, 10, 15]

plt.figure(figsize=(8, 8))

allAmplitudes = []
allAngles = []

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq'] == freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)

    data = data.iloc[goodIdx]

    complexVals = data['complex'].astype(complex)
    amplitudes = data['stim_amp']

    baselineComplex = getBaselineComplex(baseline, freq)

    complexVals = np.insert(complexVals, 0, baselineComplex)
    amplitudes = pd.concat([pd.Series([0]), amplitudes], ignore_index=True)

    angles = np.angle(complexVals) * 180 / np.pi

    allAngles.append(np.array(angles))
    allAmplitudes.append(np.array(amplitudes))

singleCellPhase = -34.79

# --- Build grouped bar plot ---
all_amp_values = np.sort(np.unique(np.concatenate(allAmplitudes)))
group_spacing = 1.8
x = np.arange(len(all_amp_values)) * group_spacing

bar_width = 0.25

for j, freq in enumerate(stimFreqs):
    # Map amplitudes to angles for this frequency
    amp_to_angle = dict(zip(allAmplitudes[j], allAngles[j]))

    # Fill missing amplitudes with NaN
    y = [amp_to_angle.get(a, np.nan) for a in all_amp_values]

    plt.bar(
        x + (j - (len(stimFreqs)-1)/2) * bar_width,
        y,
        width=bar_width,
        label=f'{freq} Hz Stimulation'
    )

# Horizontal reference line
plt.axhline(
    singleCellPhase,
    color='k',
    linestyle='--',
    linewidth=2,
    label='Single-cell phase delay'
)

plt.xticks(x, all_amp_values)
plt.xlabel('Stimulus Amplitude (V/m)')
plt.ylabel('Phase Delay (deg)')
plt.legend()

plt.tight_layout()
plt.savefig('output/angles.png', dpi=600)
plt.show()

In [ ]:
stimFreqs = [7, 10, 15]

plt.figure(figsize=(8, 8))

allAmplitudes = []
allAngles = []

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq'] == freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)

    data = data.iloc[goodIdx]

    complexVals = data['complex'].astype(complex)
    amplitudes = data['stim_amp']

    baselineComplex = getBaselineComplex(baseline, freq)

    complexVals = np.insert(complexVals, 0, baselineComplex)
    amplitudes = pd.concat([pd.Series([0]), amplitudes], ignore_index=True)

    angles = np.angle(complexVals) * 180 / np.pi

    allAngles.append(np.array(angles))
    allAmplitudes.append(np.array(amplitudes))

# allAngles = np.array(allAngles)
# allAmplitudes = np.array(allAmplitudes)
singleCellPhase = -34.79

# --- Build grouped bar plot ---


bar_width = 0.25

for i in range(len(stimFreqs)):
    plt.plot(allAmplitudes[i],allAngles[i],c='C'+str(i),label=f'{stimFreqs[i]} Hz Stimulation')
    plt.scatter(allAmplitudes[i],allAngles[i],s=(allAmplitudes[i]+.01)*20,c='C'+str(i))

# Horizontal reference line
plt.axhline(
    singleCellPhase,
    color='k',
    linestyle='--',
    linewidth=2,
    label='Single-cell phase delay'
)

plt.xlabel('Stimulus Amplitude (V/m)')
plt.ylabel('Phase Delay (deg)')
plt.legend()

plt.tight_layout()
plt.savefig('output/angles.png', dpi=600)
plt.show()

In [ ]:
stimFreqs = [7, 10, 15]

plt.figure(figsize=(8, 8))

allAmplitudes = []
allAngles = []

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq'] == freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)

    data = data.iloc[goodIdx]

    complexVals = data['complex'].astype(complex)
    amplitudes = data['stim_amp']

    baselineComplex = getBaselineComplex(baseline, freq)

    complexVals = np.insert(complexVals, 0, baselineComplex)
    amplitudes = pd.concat([pd.Series([0]), amplitudes], ignore_index=True)

    angles = np.angle(complexVals) * 180 / np.pi

    allAngles.append(np.array(angles))
    allAmplitudes.append(np.array(amplitudes))

# allAngles = np.array(allAngles)
# allAmplitudes = np.array(allAmplitudes)
singleCellPhases = [-31,-36,-44]

# --- Build grouped bar plot ---


bar_width = 0.25

for i in range(len(stimFreqs)):
    plt.plot(allAmplitudes[i],allAngles[i]-singleCellPhases[i],c='C'+str(i),label=f'{stimFreqs[i]} Hz Stimulation')
    plt.scatter(allAmplitudes[i],allAngles[i]-singleCellPhases[i],s=(allAmplitudes[i]+.01)*20,c='C'+str(i))

plt.xlabel('Stimulus Amplitude (V/m)')
plt.ylabel('Difference between column phase delay\nand single-neuron phase delay (degrees)')
plt.legend()

plt.tight_layout()
plt.savefig('output/angles.png', dpi=600)
plt.show()

In [ ]:
allAngles

In [ ]:
stimFreqs = [7,10,15]

plt.figure(figsize=(8,8))

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']
    plv = data['plv']

    amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
    plv = pd.concat([pd.Series(getBaselinePLV(baseline,freq)),plv],ignore_index=True)

    plt.plot(amplitudes,plv,'-',label=f'{freq} Hz')
    plt.scatter(amplitudes,plv,s=amplitudes*20,c='C'+str(j))

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('PLV')
plt.legend()
plt.tight_layout()
plt.savefig('output/plv.png',dpi=600)

In [ ]:
stimFreqs = [7,10,15]

plt.figure()

for j, freq in enumerate(stimFreqs):
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    amplitudes = data['stim_amp']
    dfr = data['deltafr']
    power = data['power_norm']

    plt.plot(amplitudes,dfr,'-',label=f'{freq} Hz')
    plt.scatter(amplitudes,dfr,s=amplitudes*20,c='C'+str(j))

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel(r'$\Delta$ FR')
plt.legend()
plt.tight_layout()
plt.savefig('output/dfr.png',dpi=600)

In [ ]:
cellTypes = ['PYR','INT_EXC','INT_INH']
colors = ['r','brown','cyan']
markers = ['^','s','X']

plt.figure()

stimFreq = 10

data = simulations[simulations['stim_freq']==stimFreq]

goodIdx = []
for i in range(len(data)):
    if 'highCorrelation' not in data.iloc[i]['sim_id']:
        goodIdx.append(i)

data = data.iloc[goodIdx]

amplitudes = data['stim_amp']

power = data['power_norm']*data['total_power']

# amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
# power = pd.concat([pd.Series(getBaselinePower(baseline,stimFreq)),power],ignore_index=True)

plt.plot(amplitudes,power,'-',color='C1',label=f'All cells')
plt.scatter(amplitudes,power,s=amplitudes*20,c='C1',)

for j, celltype in enumerate(cellTypes):
    
    
    power = data['total_power_'+celltype]
    # power = pd.concat([pd.Series(getBaselinePower(baseline,stimFreq,celltype)),power],ignore_index=True)

    plt.plot(amplitudes,power,'-',color=colors[j],label=f'{celltype}')
    plt.scatter(amplitudes,power,s=amplitudes*20,c=colors[j],marker=markers[j])

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('Normalized power at target frequency')
plt.legend()
plt.tight_layout()
# plt.savefig('output/power_by_type.png',dpi=600)

In [ ]:
cellTypes = ['PYR','INT_EXC','INT_INH']
colors = ['r','brown','cyan']
markers = ['^','s','X']

plt.figure()

stimFreq = 10

data = simulations[simulations['stim_freq']==stimFreq]

goodIdx = []
for i in range(len(data)):
    if 'highCorrelation' not in data.iloc[i]['sim_id']:
        goodIdx.append(i)

data = data.iloc[goodIdx]

amplitudes = data['stim_amp']

power = data['power_norm']

amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)
power = pd.concat([pd.Series(getBaselinePower(baseline,stimFreq)),power],ignore_index=True)

plt.plot(amplitudes,power,'-',color='C1',label=f'All cells')
plt.scatter(amplitudes,power,s=amplitudes*20,c='C1',)

for j, celltype in enumerate(cellTypes):
    
    
    power = data['power_norm_'+celltype]
    power = pd.concat([pd.Series(getBaselinePower(baseline,stimFreq,celltype)),power],ignore_index=True)

    plt.plot(amplitudes,power,'-',color=colors[j],label=f'{celltype}')
    plt.scatter(amplitudes,power,s=amplitudes*20,c=colors[j],marker=markers[j])

plt.xlabel('Stimulus amplitude (V/m)')
plt.ylabel('Normalized power at target frequency')
plt.legend()
plt.tight_layout()


In [ ]:
cellTypes = ['INT_EXC','INT_INH','PYR']
colors = ['brown','cyan','r',]
markers = ['s','X','^']

stimFreq = 10

plt.figure(figsize=(8,8))
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)

allDataReal = np.real(simulations['complex'].astype(complex))
allDataImag = np.imag(simulations['complex'].astype(complex))

plotLim = 8

data = simulations[simulations['stim_freq']==stimFreq]

goodIdx = []
for i in range(len(data)):
    if 'highCorrelation' not in data.iloc[i]['sim_id']:
        goodIdx.append(i)

data = data.iloc[goodIdx]

for j, celltype in enumerate(cellTypes):
    
    complexvals = data['complex_'+celltype]

    realParts = np.real(complexvals.astype(complex))
    imagParts = np.imag(complexvals.astype(complex))

    realParts = np.insert(realParts,0,np.real(getBaselineComplex(baseline,stimFreq,celltype)))
    imagParts = np.insert(imagParts,0,np.imag(getBaselineComplex(baseline,stimFreq,celltype)))

    plt.plot(realParts,imagParts,'-',color=colors[j],label=f'{celltype}')
    plt.scatter(realParts,imagParts,s=(amplitudes+.01)*20,c=colors[j],marker=markers[j])


amplitudes = data['stim_amp']
amplitudes = pd.concat([pd.Series(0),amplitudes],ignore_index=True)

realParts = np.real(data['complex'].astype(complex))
imagParts = np.imag(data['complex'].astype(complex))

realParts = np.insert(realParts,0,np.real(getBaselineComplex(baseline,stimFreq)))
imagParts = np.insert(imagParts,0,np.imag(getBaselineComplex(baseline,stimFreq)))

plt.plot(realParts,imagParts,'-',c='C1',label=f'All cells')
plt.scatter(realParts,imagParts,s=amplitudes*20,c='C1')

from matplotlib.lines import Line2D

celltype_handles = [
    Line2D(
        [0], [0],
        color=colors[j],
        marker=markers[j],
        linestyle='-',
        markersize=8,
        label=cellTypes[j]
    )
    for j in range(len(cellTypes))
]

allcells_handle = Line2D(
    [0], [0],
    color='C1',
    linestyle='-',
    marker='o',
    markersize=8,
    label='All cells'
)

freq_legend = plt.legend(
    handles= celltype_handles + [allcells_handle],
    loc='upper right',
    ncol=2
)

size_handles = [
    plt.scatter([], [], s=(a+.01) * 20, color='grey')
    for a in amplitudes
]

size_legend = plt.legend(
    size_handles,
    [f'{a}' for a in amplitudes],
    title='Stimulus amplitude (V/m)',
    loc='lower left',
    ncol=2
)

plt.gca().add_artist(size_legend)
plt.gca().add_artist(freq_legend)

plt.xlabel('Real part')
plt.ylabel('Imaginary part')
plt.xlim([-4,8])
plt.ylim([-8,4])
plt.tight_layout()
plt.title('10 Hz stimulus')
plt.savefig('output/complex_by_type.png',dpi=600)


In [ ]:
np.angle(data['complex'].astype(complex).iloc[-1])*180/np.pi

In [ ]:
stimFreqs = [7,10,15]

amplitudes = np.empty((3,4),dtype=complex)
firsthalves = np.empty((3,4),dtype=complex)
secondhalves = np.empty((3,4),dtype=complex)

plt.figure()

for j, freq in enumerate(stimFreqs):
    
    data = simulations[simulations['stim_freq']==freq]

    goodIdx = []
    for i in range(len(data)):
        if 'highCorrelation' not in data.iloc[i]['sim_id']:
            goodIdx.append(i)
    
    data = data.iloc[goodIdx]

    for i in range(len(data)):

        amplitudes[j,i] = data.iloc[i]['stim_amp']

        config_path = data.iloc[i]['source_file']

        simulationData, allData, cellData, spikeData = load_data(config_path)
        cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)

        hist_firsthalf, _ = get_firingrate_hist(
                    simulationData, spikeData, bin_width=5, n_neurons=cell_to_rank.shape[0], delay=1500 , time_stop=2000, smoothed=True)
        
        xf_pos_1, _, yf_1 = compute_fourier(hist_firsthalf, 5 * 1e-3, referenced=True, sim_id=data.iloc[i]['sim_id'])

        target_1 = yf_1[np.argmin(np.abs(xf_pos_1-freq))]

        firsthalves[j,i] = target_1

        hist_secondhalf, _ = get_firingrate_hist(
                    simulationData, spikeData, bin_width=5, n_neurons=cell_to_rank.shape[0], delay=2000 , time_stop=2500, smoothed=True)
        
        xf_pos_2, _, yf_2 = compute_fourier(hist_secondhalf, 5 * 1e-3, referenced=True, sim_id=data.iloc[i]['sim_id'])

        target_2 = yf_2[np.argmin(np.abs(xf_pos_2-freq))]

        secondhalves[j,i] = target_2



In [ ]:
amplitudes[j]

In [ ]:
amplitudes = np.abs(amplitudes)

plt.figure(figsize=(6,3))

plt.subplot(1,2,1)
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlim([-5,5])
plt.ylim([-5,5])
for j, freq in enumerate(stimFreqs):

    plt.plot(np.real(firsthalves[j]),np.imag(firsthalves[j]),'-',c='C'+str(j),label=f'{freq} Hz')
    plt.scatter(np.real(firsthalves[j]),np.imag(firsthalves[j]),s=amplitudes[j]*20,c='C'+str(j))

plt.subplot(1,2,2)
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlim([-5,5])
plt.ylim([-5,5])
for j, freq in enumerate(stimFreqs):

    plt.plot(np.real(secondhalves[j]),np.imag(secondhalves[j]),'-',c='C'+str(j),label=f'{freq} Hz')
    plt.scatter(np.real(secondhalves[j]),np.imag(secondhalves[j]),s=amplitudes[j]*20,c='C'+str(j))

In [ ]:
firsthalves

In [ ]:
# I want to look at all the plv values for all simulations with 15Hz target frequency


data_15 = simulations[simulations['stim_freq'] == 15]
plv_values = data_15['plv'].values
amplitude = data_15['stim_amp'].values

#Say you want to add the baseline value -> you know this is simulation sim_id = resting

#----------------------------------------
# please be careful because different machines for deployement may have different baseline values
# make sure they correspond 
# ---------------------------------------

#plv value for resting state is different based on the target frerquency of the stimulation, so make sure to use the correct one

#for 15HZ
freq = data_15['stim_freq'].values[0]
phase0 = data_15['stim_phase'].values[0]
phase_orientation = data_15['stim_orientation'].values[0]

simulationData, allData, cellData, spikeData = load_data(config_path)
cell_to_rank = geom_rank_cell_nodes(cellData, simulationData)
delay = 500 #take an initial delay of 500ms to make sure we are in the steady state of the stimulation, and not in the transient part
time_stop = simulations[simulations['sim_id'] == 'resting']['tstop'].values[0] #take the same time stop as the resting state simulation to make sure we are comparing the same time window for the plv calculation
baseline_plv15 = get_plv(
    spikeData,
    time_stop * 1e-3,
    freq,
    phase0=phase0,
    phase_orientation=phase_orientation,
    delay=delay * 1e-3,
)

plv_values = np.append(plv_values, baseline_plv15)  
amplitude = np.append(amplitude, 0)

plt.plot(amplitude, plv_values, 'o')
plt.xlabel('Amplitude')
plt.ylabel('PLV')
plt.title('PLV vs Amplitude for 15Hz stimulation')
plt.show()

# I want to do this now only for the pyramidal cells, now

cell_types_dict = extract_main_cell_types(spikeData, cellData, cell_to_rank)
spikeData_pyr, cell_to_rank_pyr = cell_types_dict['PYR']

plv_values_15_pyramidal = data_15['plv_PYR'].values
baseline_plv15_PYR = get_plv(
    spikeData_pyr,
    time_stop * 1e-3,
    freq,
    phase0=phase0,
    phase_orientation=phase_orientation,
    delay=delay * 1e-3,
)
plv_values_15_pyramidal = np.append(plv_values_15_pyramidal, baseline_plv15_PYR)
plt.plot(amplitude, plv_values_15_pyramidal, 'o')
plt.xlabel('Amplitude')
plt.ylabel('PLV (Pyramidal cells)')
plt.title('PLV vs Amplitude for 15Hz stimulation (Pyramidal cells)')
plt.show()


In [ ]:
singleCell = pd.read_csv('/mnt/c/Users/tharayil/Downloads/singleCell_15.csv')

f = 15

time = singleCell.iloc[:,0].values
soma = singleCell.iloc[:,1].values
tstart = np.argmin(np.abs(time-0.5))
soma = soma[tstart:]
time = time[tstart:]
time = time - np.min(time)

signal = np.sin(2*np.pi*f*time)

In [ ]:
time

In [ ]:
import numpy as np
from scipy.signal import correlate

# Sampling parameters
fs = len(time)/np.max(time)  # Hz

# Time vector

# Signals
x = soma
y = -signal 

# Cross-correlation
corr = correlate(y, x, mode='full')
lags = np.arange(-len(x) + 1, len(x))

# Delay estimate
lag_samples = lags[np.argmax(corr)]
delay = lag_samples / fs

# Phase difference
phase_rad = 2 * np.pi * f * delay
phase_deg = np.degrees(phase_rad)

# Wrap to [-180, 180]
phase_deg = (phase_deg + 180) % 360 - 180

print(f"Delay: {delay:.6f} s")
print(f"Phase lag: {phase_deg:.2f} degrees")

In [ ]:
(2*np.pi*f*(np.argmax(soma[:3000])-np.argmax(-signal[:3000]))/40000)*180/np.pi

In [ ]:
(31+36+44)/3

In [ ]:
plt.figure()
plt.subplot(211)
plt.plot(soma[:5000])
plt.subplot(212)
plt.plot(-signal[:5000])

In [ ]:
# FFTs
X = np.fft.rfft(soma)
Y = np.fft.rfft(signal)

fs = len(time)/np.max(time)

# Frequency bins
freqs = np.fft.rfftfreq(len(soma), d=1/fs)

# Index of frequency of interest
idx = np.argmin(np.abs(freqs - 10))

# Phase difference
phase_rad = np.angle(Y[idx]) - np.angle(X[idx])
phase_deg = np.degrees(phase_rad)

# Wrap to [-180, 180]
phase_deg = (phase_deg + 180) % 360 - 180

print(f"Phase lag: {phase_deg:.2f} degrees")